# Lab 9 — Words as Data, and What a Language Model Measures

**ECON 282E · Session 9 · November 19, 2026**

Six blocks, in the order of the decks. Each block ends with a **gate**: a bound fixed before the run.
Nothing here needs a network connection or a GPU; from a clean kernel the notebook runs in a few minutes.

| Block | min | You will | Deck |
|---|---|---|---|
| **A** | 30 | clean and cut text — one sentence step by step, Chinese segmentation, word pairs — then build a vocabulary three ways (byte-pair, WordPiece, unigram) and count: the slides' three documents by hand, TF-IDF and topics on the statements, fitted on 1994–2014 and nothing else | 9.A1 |
| **B** | 30 | train word vectors on the same window, see what word2vec does instead, what the window and the length change, and whether the slides' parallel moves survive fitting; build a hawk–dove axis, and watch it **fail** validation beside a ten-line rule that passes, read over the whole statement or one sentence | 9.A1, 9.A2 |
| **C** | 25 | see one word get two vectors; do the attention arithmetic by hand, with heads, word order and the mask; open Machine B, a small transformer trained on these statements: one layer rebuilt, its attention, its next-character distribution, its writing, its first sixty training steps; then read what a recorded encoder returns | 9.A2, 9.A3 |
| **D** | 20 | read a recorded language-model instrument: provenance, agreement, Cohen's κ, and what convergent validity does not buy | 9.B1 |
| **E** | 20 | run chapter 14's four simulations through the chapter's own code: attenuation, correction, per-class κ, self-consistency, order effects | 9.B1 |
| **F** | 25 | ask whether the Committee's words improve Session 7's bond-yield forecast, under a protocol hashed before the first scored run | 9.B1 |

**What a gate is here.** Every gate is on **procedure** or on **reproduction**, never on an outcome.
A procedure gate checks how the run was done: hashes checked before a byte is read, every fitted object
confined to the training window, the attention map causal, the test window scored once. A reproduction
gate checks that a digit some frozen artefact already carries comes back — the corpus manifest, Session 7's
published baselines, chapter 14's printed figures, the decks' worked examples, the recorded runs. A reproduction gate has **no preferred
direction**: it would pass exactly as it does now if block B's embedding signal had validated, if block D's
byte-identical repeat had agreed more than the reworded prompt, and if block F's text had improved the
forecast. Nothing in this notebook passes because a number came out in a direction the lab would like.

**One corpus, one window.** All 245 post-meeting FOMC statements, 1994-02-04 to 2026-06-17, hash-checked
against `fomc_manifest.json`. The **training window** is the 149 statements dated on or before 2014-12-31.
Every vocabulary, every IDF weight, every word vector and every scaler in this notebook is fitted on that
window; 2015 onwards is held back and only ever transformed.

The four modules behind the blocks live in `l09_assets/`: `session05_text.py` (A, B),
`session05_attention.py` (C), `session05_measure.py` (E) and `session05_forecast.py` (F). The numbers the
slides quote come from full-budget runs of exactly these functions, stored in `recorded_*.json`.
A fifth module, `l09_course.py`, runs the parts this course added (A2b to A5b, B1b to B1d, C0, C1c, C2b,
C4b, C6; B4b needs only `session05_text.py`): the four decks' worked examples, with the decks' own inputs. Where an example needs a model this
lab does not ship, it reads `recorded_deck12_examples.json`, one recorded run of those models, hash-checked
before use.

**Names inside the frozen files.** The forecast protocol, the gate file, the four modules, the recorded runs
and Machine B's files were written before this lab was numbered 9, and the protocol and Machine B's files are
hashed. So they keep their own session numbers: a hashed file cannot change by one byte, and the modules and
records name one another. Read them this way:

| In those files | In this course |
|---|---|
| Session 5; `session05_*.py`; `labs/session05_assets/` | this lab and Session 9's decks; `labs/l09_assets/` |
| Deck 5.1, 5.2, 5.3; Deck 5.6 | 9.A1, 9.A2, 9.A3; 9.B1, the walk-through of this lab |
| Deck 5.4, 5.5 | not presented in this course; 9.B1 explains blocks D and E |
| Session 2; `session02_published`; `labs/session02_assets/` | Session 7's bond-yield forecast (7.B1) and its two frozen files, copied to `l09_assets/s07/` |
| Session 1; `session01_demo.py`; "Session 1's Machine B" | Machine B, the frozen model block C opens, and its training run |

Any other session number in those files belongs to the same earlier numbering. The cells below use this
course's numbers.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")       # set before numpy is imported; shared machines need it
%matplotlib inline
import collections, itertools, json, sys, textwrap, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAB = Path.cwd() if (Path.cwd() / "l09_assets").is_dir() else Path.cwd() / "labs"
ASSETS = LAB / "l09_assets"
sys.path.insert(0, str(ASSETS))
import session05_text as tx            # blocks A, B
import session05_attention as at       # block C
import session05_measure as me         # block E (and block D's kappa)
import session05_forecast as fc        # block F
import l09_course as cx                # the parts this course added: the decks' worked examples

GATE = json.loads((ASSETS / "lab_gates.json").read_text(encoding="utf-8"))
REC = {k: json.loads((ASSETS / f"recorded_{k}.json").read_text(encoding="utf-8"))
       for k in ("text", "attention", "measure", "forecast")}
TEAL, RED, MUTED = "#006D77", "#A3473B", "#5C686B"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
T_START = time.time()
GATES = []

def gate(name, ok, detail=""):
    GATES.append((name, bool(ok)))
    print(f"GATE {name}: {'pass' if ok else 'FAIL'}   {detail}")
    assert ok, f"gate {name} failed: {detail}"

print("numpy", np.__version__, "| pandas", pd.__version__, "| recorded runs:", ", ".join(REC))

# Block A — Tokens: what the computer actually reads

A statement is a string of characters. No regression takes a string. Between the two sits a **vocabulary**:
a finite list of units, plus a rule that turns any document into counts over that list. Block A builds one
twice — once by merging characters upward (byte-pair encoding) and once by counting words and weighting
them (TF-IDF) — and fits both on the 1994–2014 window alone.

## A1. The corpus, and the line down the middle of it

**Training window**: the statements dated on or before 2014-12-31. **Held back**: everything after.
The line is not decoration. A vocabulary fitted on the whole sample has seen the words of 2020 when it
scores 1994, and every later claim about out-of-sample behaviour is then about a sample that was never out.

In [ ]:
statements = tx.load_corpus()                      # every file's sha256 checked against fomc_manifest.json
train = tx.training_window(statements)             # dated on or before 2014-12-31, and nothing else
held_back = [s for s in statements if s.date > tx.TRAIN_END]

for label, block in (("all", statements), ("training window", train), ("held back", held_back)):
    d = tx.corpus_summary(block)
    print(f"{label:<16} n = {d['n']:>3}   {d['first']} to {d['last']}   {d['characters']:>7,} characters"
          f"   {d['words']:>6,} words   median statement {d['median_characters']:>5,} characters")

per_year = REC["text"]["corpus"]["per_year_1994_1999"]
print("\nstatements per year, 1994-1999:  " + "   ".join(f"{y}: {n}" for y, n in per_year.items()))
print("The Committee met eight times a year throughout. It did not publish a statement after every")
print("meeting until 1999, so the corpus is what was released, not what was decided.")
print(f"\nmedian statement length: {tx.corpus_summary(train)['median_characters']:,} characters in the"
      f" training window against {tx.corpus_summary(held_back)['median_characters']:,} after it —")
print("the object being measured changed shape over the sample, before anyone measured anything.")

ok = (len(statements) == GATE["corpus_statements"]
      and len(train) == GATE["corpus_training_statements"]
      and len(held_back) == GATE["corpus_held_back_statements"]
      and max(s.date for s in train) <= GATE["corpus_training_end"]
      and statements[0].date == GATE["corpus_first"] and statements[-1].date == GATE["corpus_last"])
gate("A1", ok, f"{len(statements)} statements, all hashes match; {len(train)} of them in the window, "
               f"the last dated {max(s.date for s in train)}")

## A2. Five switches, none of them forced by the data

Lower-casing, dropping numbers, dropping stop-words, stemming, dropping one-character tokens; the tokenizer
drops punctuation in every row. Each is a defensible choice; none is implied by the corpus. Every row below
is the **same 149 statements** with one more switch on than the row above, so the spread across rows is the size of the researcher's hand before a single number is reported.

In [ ]:
ladder = tx.degrees_of_freedom(train)
print(ladder[["tokens", "vocabulary", "vocabulary_vs_baseline", "top_terms"]].to_string())
print(f"\nvocabulary runs from {ladder['vocabulary'].min():,} to {ladder['vocabulary'].max():,} "
      f"— a factor of {ladder['vocabulary'].max() / ladder['vocabulary'].min():.1f} — on one corpus.")
print("the last row's chain:", ladder['steps'].iloc[-1])
print("\nThe stop-word list is short and auditable, and 'not', 'no' and 'less' are deliberately absent:")
print(f"{len(tx.STOPWORDS)} words, and negation carries policy meaning.")

## A2b. One sentence, cleaned a step at a time

A2 measured the switches on the whole window. Here they run on one sentence, so what each step deletes can be
read: the decision sentence of the 2004-06-30 statement, the meeting that opened the 2004–06 tightening cycle.
Then what this lab's stop list does to the words 9.A1's "Stop Words" frame says to keep; stems cut by rule against
lemmas looked up in a dictionary, and the Chinese verb that needs neither ("Stems, Lemmas"); and the three raw
strings of "Cleaning, Worked on One Sentence": mark-up and an entity to strip; traditional characters, without
which 通脹 and 通胀 are two words; full-width letters and digits, without which ＣＰＩ never matches CPI.

In [ ]:
s = next(x for x in statements if x.date == "2004-06-30")
for label, value, note in cx.seven_steps(tx.decision_sentence(s.text)):
    print(f"{label:<27}{value if isinstance(value, str) else ' · '.join(value)}")
    if note:
        print(f"{'':<27}{note}")
print("\nStep 3 deletes the size of the move and the new level; step 5, the words that tie the rest together;")
print("step 6 cuts by rule, so 'decided' becomes 'decid' and 'basis' becomes 'basi'.")

counts = tx.word_frequencies(train)
print("\n9.A1's 'Stop Words' frame keeps five words. How often the window uses each, and what this lab's list does:")
print("   " + ", ".join(f"{w} {counts[w]} ({'dropped' if w in tx.STOPWORDS else 'kept'})" for w in cx.SIGNAL_WORDS))
dropped = [w for w in cx.SIGNAL_WORDS if w in tx.STOPWORDS]
also = [w for w in ("above", "below") if w in tx.STOPWORDS]
print(f"The list keeps 'not', 'no' and 'less' on purpose (A2), yet drops {', '.join(dropped)}, and {' and '.join(also)}")
print("too: direction words a general list deletes. Read a stop list before using it.")

stems = dict(zip(cx.STEM_PROBES, tx.crude_stem(list(cx.STEM_PROBES))))
print("\nstems cut by rule: " + ", ".join(f"{w} -> {stem}" for w, stem in stems.items()))
print(f"Porter's stemmer, on the slide, gives 'studi'; this lab's cruder rule gives '{stems['studies']}'. Either way a")
print(f"non-word, and 'inflation' becomes '{stems['inflation']}'.")
print("\nlemmas looked up: " + ", ".join(f"{w} -> {lemma}" for w, lemma in cx.LEMMAS.items()))
print("A lemma is a dictionary word, and better -> good is beyond any suffix rule. The price is the dictionary and")
print("the part of speech: 'housing' is read here as a noun; as a verb its lemma is 'house'.")
print("Chinese needs neither: 上调 (raise) keeps its form, and the tense is a word of its own.")
for text, gloss in cx.ZH_TENSE:
    print(f"   {text}   {' | '.join(cx.max_match(text, cx.ZH_DICTIONARY, backward=True))}   {gloss}")

print("\n9.A1's three raw strings, three steps each:")
for name, raw in cx.RAW_STRINGS:
    print(name)
    for label, text in cx.clean_raw(raw):
        print(f"   {label:<33}{text}")
print("Each step is a rule in a script; archive the script with the data.")

## A2c. Where Chinese words begin

Chinese text has no spaces, so a segmenter decides after each character whether a word ends there.
**Maximum matching** is the dictionary half of segmenters such as Jieba, THULAC and LTP: take the longest
dictionary word that fits, scanning from the left (forward) or from the right (backward); a character no entry
covers stands alone. The real tools add a statistical labeller for the words a dictionary lacks. The cell runs
the dictionary half alone on 9.A1's examples, with a small general dictionary, then adds the frame's **user
dictionary** of domain compounds.

In [ ]:
print(f"a general dictionary of {len(cx.ZH_DICTIONARY)} entries\n")
for text, gloss in cx.ZH_EXAMPLES:
    bwd = cx.max_match(text, cx.ZH_DICTIONARY, backward=True)
    print(f"{text}   {gloss}")
    print(f"   forward    {' | '.join(cx.max_match(text, cx.ZH_DICTIONARY))}")
    print(f"   backward   {' | '.join(bwd)}")
    print("   labels     " + " ".join(f"{ch}{tag}" for ch, tag in cx.bmes(bwd)))
print("\nThe labels are read off the backward cut, one per character: B begins a word, M is inside it, E ends it,")
print("S stands alone. A statistical segmenter predicts these labels; the dictionary only proposes them.")
print("On 研究生命科学, forward matching takes 研究生 (graduate student) and strands 命; backward finds 研究 | 生命科学.")
print("On 行业政策支持 both cuts are sensible, with different head words, as 9.A1 says: support for an industry")
print("policy, or policy support for an industry. Whether 支持 is counted on its own depends on which one ran.\n")

user = cx.ZH_DICTIONARY | cx.USER_DICTIONARY
print("the user dictionary: " + "   ".join(sorted(cx.USER_DICTIONARY)))
for text, gloss in cx.ZH_USER_EXAMPLES:
    print(f"{text}   {gloss}")
    print(f"   general dictionary   {' | '.join(cx.max_match(text, cx.ZH_DICTIONARY))}")
    print(f"   + user dictionary    {' | '.join(cx.max_match(text, user))}")
print("Without the entry, 货币 (currency) is cut away from the committee it names. Add the domain terms before")
print("counting, and publish the list with the counts.")

## A2d. Pairs as features: n-grams

A bigram counts two adjacent tokens as one feature, so *less accommodative* gets a column of its own instead of
adding one more *accommodative*. The price is columns: $V$ words allow $V^2$ pairs. Which pairs count is a rule
the researcher sets and reports; here it is the rule A4's vocabulary uses, a pair in at least two statements of
the window.

In [ ]:
ng = cx.ngram_counts(train, min_df=GATE["tfidf_min_df"])
print(f"training window: {ng['n_statements']} statements, the baseline tokenizer")
print(f"   words   V = {ng['V']:,}; in at least {ng['min_df']} statements: {ng['unigrams_min_df']:,}")
print(f"   pairs   V^2 = {ng['V_squared']:,} possible; {ng['bigrams_seen']:,} seen; in at least {ng['min_df']} "
      f"statements: {ng['bigrams_min_df']:,}")
print(f"   columns under that rule: {ng['unigrams_min_df']:,} for words alone, "
      f"{ng['unigrams_min_df'] + ng['bigrams_min_df']:,} for words and pairs\n")
print("pairs that open with not, no or less, and how often each occurs:")
print("   " + ", ".join(f"{pair} {n}" for pair, n in ng["negation_bigrams"][:12]))
acc, less_acc = ng["unigram_counts"]["accommodative"], ng["bigram_counts"][("less", "accommodative")]
print(f"\n'accommodative', one of B2's dovish seed words, occurs {acc} times, {less_acc} of them right after 'less'.")
print(f"A word column counts those {less_acc} as dovish; a pair column keeps them apart.")
print(f"'sufficient' occurs {ng['unigram_counts']['sufficient']} times and 'not sufficient' "
      f"{ng['bigram_counts'][('not', 'sufficient')]} times: the slide's pair need not be in your corpus.")

## A3. Byte-pair encoding, learned here

A word-level vocabulary has a hard edge: a word it never saw has no column at all. Byte-pair encoding
removes the edge by building the vocabulary from **below**. Start with characters plus an end-of-word marker;
count adjacent symbol pairs, weighted by word frequency; glue the commonest pair everywhere; repeat.
After 200 rounds the frequent words are single symbols and everything else is spelled out of pieces.

The merges below are learned on the training window. Nothing after 2014 contributed a single count.

In [ ]:
merges = tx.learn_bpe(train, n_merges=GATE["bpe_merges"])       # learned here, from the training window
print(f"{len(merges)} merges, in the order they were learned. The first "
      f"{GATE['bpe_first_merges_checked']}:")
for start in range(0, GATE["bpe_first_merges_checked"], 5):
    print("   " + "   ".join(f"{k + 1:>2}. {a}+{b}"
                             for k, (a, b) in enumerate(merges[start:start + 5], start=start)))
print("\nThe first merges are the grammar of English, discovered from counts: the end-of-word marker glues")
print("onto 'e' and 's' before anything else, and 'th' + 'e</w>' becomes the word 'the' by merge 11.")

In [ ]:
words = tx.word_frequencies(train)                  # the word-level vocabulary of the same window
probes = GATE["bpe_out_of_vocabulary_probes"]
print(f"the word-level vocabulary of the training window: {len(words):,} distinct words\n")
print(f"{'word':<18}{'word-level column?':<22}byte-pair segmentation")
for w in ("raising", "unemployment", "accommodation", *probes):
    pieces = tx.apply_bpe(w, merges)
    print(f"{w:<18}{'yes' if w in words else 'NO — no column at all':<22}" + " | ".join(pieces))
print("\n'reflationary' and 'disinflationary' are real words of monetary economics that this corpus never")
print("uses. A word-level model drops them on the floor. Byte-pair spells them out of pieces it has seen,")
print("and 'infl' and 'tion' carry the parts of the meaning that a count model can still use.")

ok = ([f"{a}+{b}" for a, b in merges[:GATE["bpe_first_merges_checked"]]]
      == REC["text"]["bpe"]["first_merges"]
      and len(words) == GATE["bpe_word_level_vocabulary"]
      and all(w not in words and len(tx.apply_bpe(w, merges)) >= GATE["bpe_min_pieces_for_an_unseen_word"]
              for w in probes))
gate("A2", ok, f"the first {GATE['bpe_first_merges_checked']} merges reproduce the recorded run; "
               f"both probes are outside the {len(words):,}-word vocabulary and segment into pieces")

## A3b. Three ways to learn the pieces

A3 learned byte-pair merges: glue the commonest adjacent pair. 9.A1's two other schemes change the rule.
**WordPiece** glues the pair seen together far more often than its parts' own counts predict, the score
$c(ab)/\bigl(c(a)\,c(b)\bigr)$, and cuts a word by taking the longest known piece from the left, then the
longest piece that continues it (`##`). **Unigram** starts from many candidate pieces, each with a probability,
and keeps dropping the ones the corpus would miss least; it cuts a word the most probable way (`▁` opens a word).

Both are learned below on the window's alphabetic words, on a budget like A3's: 200 merges for WordPiece, and
as many pieces for unigram as WordPiece ends with. Both are simplified: BERT's WordPiece and SentencePiece's
unigram are trained on far larger corpora, and SentencePiece weighs every split of a word where this version
keeps only the best. Then the phrase of "Three Ways, One Phrase", cut by the three schemes learned here and by
three real vocabularies, read from the recorded run; last, the two words of "Why Subwords", cut by A3's merges.

In [ ]:
alpha = cx.alphabetic(words)                     # A3's word counts, alphabetic words only
t0 = time.time()
wp_merges = cx.learn_wordpiece(alpha, n_merges=GATE["bpe_merges"])
wp_vocab = cx.wordpiece_vocabulary(alpha, wp_merges)
uni = cx.learn_unigram(alpha, target=len(wp_vocab))
print(f"{len(alpha):,} alphabetic words. WordPiece: {len(wp_merges)} merges, {len(wp_vocab)} pieces. Unigram: "
      f"{uni['rounds'][0]['pieces']:,} candidate pieces in use, pruned to {uni['vocabulary']} over "
      f"{len(uni['rounds']) - 1} rounds. {time.time() - t0:.0f} s")
print("byte-pair's first merges (the commonest pair):  " + ", ".join(f"{a}+{b}" for a, b in merges[:6]))
print("WordPiece's first merges (the score):            " + ", ".join(f"{a}+{b}" for a, b, _ in wp_merges[:6]))
print("Byte-pair starts with what is common, letters at the end of a word. WordPiece starts with letters that")
print("almost never appear apart, such as b and j in 'objective' and q and u.")

record = cx.load_record()                        # its sha256 is checked before a byte is used
sch = record["schemes"]
phrase = cx.SCHEME_PHRASE
cuts = [("byte-pair, learned in A3", "lab-bpe", [p for w in phrase.split() for p in tx.apply_bpe(w, merges)]),
        ("WordPiece, learned here", "wordpiece", [p for w in phrase.split() for p in cx.wordpiece_cut(w, wp_vocab)]),
        ("unigram, learned here", "sentencepiece", [p for w in phrase.split() for p in cx.unigram_cut(w, uni)])]
cuts += [(f"{scheme}, {sch[key]['tokenizer']}", key, sch[key]["pieces"])
         for scheme, key in (("byte-pair", "bpe"), ("WordPiece", "wordpiece"), ("unigram", "sentencepiece"))]
print(f'\n"{phrase}"   ({cx.SPACE} is a space carried inside a piece; </w> closes a word in the merges of A3)')
for name, _, pieces in cuts:
    print(f"   {name}: {len(pieces)} pieces")
    print("      " + " | ".join(p.replace(" ", cx.SPACE) for p in pieces))
whole = [name for name, _, pieces in cuts if any("inflation" in p for p in pieces)]
print(f"\nkeeps 'inflation' whole inside 'disinflationary': {'; '.join(whole) if whole else 'none of the six'}")
print("\n9.A1's 'Why Subwords', cut by A3's byte-pair merges:")
for w in cx.SUBWORD_PROBES:
    print(f"   {w:<16}{words.get(w, 0)} times in the window   " + " | ".join(tx.apply_bpe(w, merges)))
print("'macroeconomics' is not among the window's words, yet the merges spell it from pieces the window does use.")
print("No merge learned from English touches 专精特新 (specialised, refined, distinctive, innovative firms), so")
print("each character stands alone, as on the slide. This toy replays merges over any character it is given; a")
print("character vocabulary learned from English would have no entry for them at all. A byte-level vocabulary such")
print("as cl100k_base starts from the 256 bytes, so every string, in any script, has a representation.")

learned = (("byte-pair", {a + b for a, b in merges}), ("WordPiece", wp_vocab), ("unigram", set(uni["logp"])))
print("the longest pieces each rule built, from the same 149 statements:")
for name, pieces in learned:
    bare = sorted(pieces, key=lambda p: (-len(p.replace("##", "").replace("</w>", "").replace(cx.MARK, "")), p))
    print(f"   {name:<11}" + ", ".join(bare[:6]))
spelled_out = [w for w in phrase.split() if len(cx.wordpiece_cut(w, wp_vocab)) == len(w)]
print("Byte-pair's longest pieces are the corpus's commonest words. WordPiece's are rare words, glued letter by")
print("letter because their letters seldom appear apart.")
if spelled_out:
    print(f"After {len(wp_merges)} merges it still spells " + " and ".join(f"'{w}'" for w in spelled_out)
          + " letter by letter.")
print("Unigram keeps frequent whole words and word bodies. Same text, a budget of about 250 pieces each: each rule")
print("spends it differently, so the same word is cut differently. The pieces follow counts, not the parts of the")
print("word (9.A1); the three real vocabularies were learned on web-scale text, with far larger budgets.")

spelled = [cx.spells_back(pieces, phrase, scheme) for _, scheme, pieces in cuts]
n_pieces = {k: len(sch[k]["pieces"]) for k in cx.SLIDE_PIECE_COUNTS}
gate("A3b", all(spelled) and n_pieces == cx.SLIDE_PIECE_COUNTS,
     "all six cuts spell the phrase back once their markers are removed; the recorded cuts have "
     f"{n_pieces['bpe']}, {n_pieces['wordpiece']} and {n_pieces['sentencepiece']} pieces, as 9.A1 prints")

## A4. TF-IDF, first on eight sentences a reader can check by hand

$\mathrm{tf}_{d,w}$ is the share of document $d$'s tokens that are $w$. $\mathrm{idf}_w = \log(N / \mathrm{df}_w)$
where $\mathrm{df}_w$ counts the **fitting** documents containing $w$. Their product is large when a word is
frequent *here* and rare *elsewhere* — which is what "what this meeting said that others did not" means.

The same two formulas run over 245 statements below. First they run over eight sentences, so the arithmetic
is visible rather than asserted.

In [ ]:
toy = tx.toy_tfidf()
for name, sentence in toy["sentences"].items():
    print(f"{name}   {sentence}")
w = toy["weights"]
print()
print("document frequency and IDF, the four commonest terms and the four rarest:")
print(pd.concat([w.sort_values("documents", ascending=False).head(4),
                 w.sort_values("documents").head(4)]).to_string())
print("\nthe three heaviest terms of each sentence:")
print(toy["top_terms"].drop(columns="sentence").to_string())
print(f"\n'the' appears in {int(w.loc['the', 'documents'])} of 8 sentences, so its IDF is "
      f"{w.loc['the', 'IDF']:.3f} and it can never be anyone's top term.")
print(f"cosine(d1, d2) on raw counts is {toy['cosine_counts'].iloc[0, 1]:.3f}; on TF-IDF it is "
      f"{toy['cosine_tfidf'].iloc[0, 1]:.3f}.")
print("d1 cuts rates to support growth and d2 raises them to curb inflation. On counts they are close,")
print("because they share every function word. Weighting by rarity is what pulls them apart.")

Now the corpus. `fit_tfidf` is handed the 149 training statements and nothing else; `transform` is then
applied to all 245. The cost of not peeking is visible in the last two lines: about one token in ten of a
held-back statement falls outside the fitted vocabulary and is simply dropped.

In [ ]:
model = tx.fit_tfidf(train, min_df=GATE["tfidf_min_df"])     # vocabulary and IDF: the window only
X = model.transform(statements)                              # applied to all 245

print(f"vocabulary {len(model.vocabulary):,} terms (min_df = {model.min_df}, bare numerals dropped), "
      f"fitted on {model.n_fit} documents")
print(f"matrix {X.shape[0]} x {X.shape[1]}, {100 * (X == 0).mean():.1f}% of its entries zero")
print(f"token coverage: {100 * model.coverage(train):.1f}% inside the training window, "
      f"{100 * model.coverage(held_back):.1f}% on the held-back statements\n")

for date in ("2008-12-16", "2020-03-15", "2005-06-30"):
    row = [i for i, s in enumerate(statements) if s.date == date][0]
    print(f"{date}: " + ", ".join(tx.top_terms(X, model, row, k=8)))
era = {w: tuple(sorted({s.date[:4] for s in train if w in tx.BASELINE.run(s.text)})) for w in ("fulfill", "obligation", "removed")}
spans = {}
for w, years in era.items():
    spans.setdefault(years, []).append(w)
both = lambda xs: xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1]
print("\nThe heaviest terms are what is rare in the window. Some are the news of the meeting (2008, 2020); some are")
print("wording the Committee used for a year or two ("
      + "; ".join(f"{both(ws)} only in {both(list(y))}" for y, ws in spans.items()) + "),")
print("and some are names on the roster (cumming, chair). TF-IDF weighs down what every statement shares, which is")
print("the whole claim it makes and the reason the representation is worth more than a raw count; it cannot tell")
print("news from a phrase that passed.")

ok = (model.n_fit == GATE["tfidf_n_fit"] and len(model.vocabulary) == GATE["tfidf_vocabulary"]
      and max(s.date for s in train) <= GATE["corpus_training_end"]
      and X.shape == (GATE["corpus_statements"], GATE["tfidf_vocabulary"]))
gate("A3", ok, f"IDF computed from {model.n_fit} documents, all dated on or before "
               f"{GATE['corpus_training_end']}; {len(model.vocabulary):,} terms; applied to all "
               f"{X.shape[0]}")

The same matrix answers a question the slides ask of three toy documents: how alike are two real statements?
Each statement is compared with the one before it, under three weightings of the same fitted vocabulary:
raw counts, presence (0/1), and TF-IDF. The same pair of documents gets three different numbers.

In [ ]:
adj = tx.adjacent_similarity(statements, model)
rows = pd.DataFrame(adj["rows"])
print(f"{adj['pairs']} consecutive pairs; mean cosine on counts {adj['mean']['counts']:.2f}, "
      f"presence {adj['mean']['binary']:.2f}, TF-IDF {adj['mean']['tfidf']:.2f}")
print(f"pairs above 0.9: counts {100 * adj['share_above_090']['counts']:.0f}%, "
      f"presence {100 * adj['share_above_090']['binary']:.0f}%, TF-IDF {100 * adj['share_above_090']['tfidf']:.0f}%\n")
for label, r in (("December 2008", adj["december_2008"]), ("lowest TF-IDF", adj["lowest_tfidf"])):
    print(f"{label}: {r['previous']} -> {r['date']}   counts {r['counts']:.2f}   presence {r['binary']:.2f}   "
          f"TF-IDF {r['tfidf']:.2f}")
print("\nOn counts nearly every statement looks like the last one: the shared template text dominates the")
print("inner product. TF-IDF weighs the template text down, so what is left is what the meeting said that")
print("the previous one did not. Same documents, three answers: the weighting is part of the measure.")

The same rule, seen from the other side. If the IDF were fitted on all 245 statements, a word the Committee
used constantly after 2014 would weigh less in every earlier row: those rows would depend on statements not
yet written. The table puts the two fits side by side.

In [ ]:
look = pd.DataFrame(tx.idf_lookahead(statements, tx.LOOKAHEAD_TERMS)).set_index("term")
print(look[["df_window", "idf_window", "df_full", "idf_full"]].round(3).to_string())
b = look.loc["balance"]
print(f"\n'balance' in a 2010 row: IDF {b['idf_window']:.2f} fitted on the window, {b['idf_full']:.2f} fitted on")
print(f"everything — {100 * (1 - b['idf_full'] / b['idf_window']):.0f}% lower, because of statements written after 2014.")
print("'pandemic' has no column at all under the window fit: the word did not exist in it.")

## A4b. The slides' three documents, by hand

9.A1 works its counting through three documents over eight words: $d_1$ = fed cut support growth,
$d_2$ = fed hike curb inflation, $d_3$ = fed hike respond inflation inflation. The cell redoes the slides'
arithmetic: one-hot vectors, the document–term matrix, TF-IDF, and the cosine under three weightings. One
difference from A4: the slides multiply the **count** by $\log(D/\mathrm{df}_v)$, while A4's `fit_tfidf`
multiplies the within-document **share**. Dividing a row by its length changes no cosine, so the two give the
same similarities at different magnitudes.

In [ ]:
td = cx.three_documents()
vocab8 = cx.TOY_VOCAB
print("the vocabulary, numbered: " + ", ".join(f"{j + 1} {w}" for j, w in enumerate(vocab8)))
print(f"one-hot vectors: o_fed . o_hike = {td['one_hot_fed_hike']:.0f}, as for any two different words\n")
print(pd.DataFrame(td["Z"].astype(int), index=["d1", "d2", "d3"], columns=vocab8).to_string())
print()
print(pd.DataFrame({"df": td["df"].astype(int), "idf = ln(3/df)": td["idf"].round(3),
                    "count in d3": td["Z"][2].astype(int), "TF-IDF in d3": td["tfidf"][2].round(3)},
                   index=vocab8).to_string())
print("\nd3 as a TF-IDF vector: (" + ", ".join(cx.as_printed(x) for x in td["tfidf"][2]) + ")")

wt = td["weightings"]
print(f"\n{'cosine':<10}{'counts':>9}{'presence':>10}{'TF-IDF':>9}")
for pair in ("d1, d2", "d2, d3", "d1, d3"):
    print(f"{pair:<10}{wt['counts'][pair]:>9.3f}{wt['presence'][pair]:>10.3f}{wt['tfidf'][pair]:>9.3f}")
print(f"\nd2 with 'raise' for 'hike', against d3, on counts: {td['cos_d2raise_d3']:.3f}, down from "
      f"{wt['counts']['d2, d3']:.3f}. Same meaning, fewer shared strings.")
share = td["Z"] / td["Z"].sum(axis=1, keepdims=True) * td["idf"]
print(f"TF-IDF built on A4's share instead of the count: cos(d2, d3) = {cx.cos(share[1], share[2]):.3f}, unchanged.")
for first, second in cx.ORDER_PAIRS:
    print(f'"{first}" and "{second}": the same bag? {cx.same_bag(first, second)}')

zero_counts = float((model.counts(statements) == 0).mean())
everywhere = [w for w, v in zip(model.vocabulary, model.idf) if v == 0.0]
print(f"\nA4's matrix of the {X.shape[0]} statements, {X.shape[0]} x {X.shape[1]}: as counts, "
      f"{100 * zero_counts:.1f}% of its cells are 0 (9.A1's sparse frame); as TF-IDF, {100 * float((X == 0).mean()):.1f}%.")
print(f"The extra zeros are the {len(everywhere)} words in all {model.n_fit} training statements ("
      + ", ".join(everywhere) + f"): IDF ln({model.n_fit}/{model.n_fit}) = 0, as for 'fed' above.")

S = cx.SLIDE_A4B
ok = (f"{100 * zero_counts:.1f}" == f"{100 * REC['text']['tfidf']['zero_share']:.1f}"
      and all(cx.printed_as(x, p) for x, p in zip(td["tfidf"][2], S["d3_tfidf"]))
      and all(cx.printed_as(td["idf"][vocab8.index(t)], p) for t, p in S["idf"].items())
      and cx.printed_as(wt["counts"]["d1, d2"], S["cos_counts"]["d1, d2"])
      and cx.printed_as(wt["counts"]["d2, d3"], S["cos_counts"]["d2, d3"])
      and cx.printed_as(td["cos_d2raise_d3"], S["cos_counts"]["d2', d3"])
      and all(cx.printed_as(wt[k]["d2, d3"], p) for k, p in S["weightings_d2_d3"].items())
      and all(cx.printed_as(wt[k]["d1, d2"], p) for k, p in S["weightings_d1_d2"].items()))
gate("A4b", ok, "9.A1's numbers come back at the precision it prints: d3 = (0, 0, 0.405, 0, 0, 0, 0.811, 1.099); "
                "cosines 0.25, 0.756 and 0.567; under three weightings 0.756, 0.75, 0.28 and 0.25, 0.25, 0; "
                f"the count matrix {100 * zero_counts:.1f}% zero, as recorded")

## A5. Topics: latent Dirichlet allocation, by collapsed Gibbs sampling

A topic model compresses a document's $V$ counts into $K$ shares. Each topic $k$ is a distribution
$\phi_k$ over words; each statement has shares $\theta_i$ over topics; each token was written by first drawing
a topic from $\theta_i$ and then a word from that topic. Fitting runs this backwards. `fit_lda_gibbs` visits
every token, removes it from the counts, and redraws its topic with probability proportional to

$$\frac{n_k^{(v)}+b}{n_k^{(\cdot)}+Vb}\times\bigl(n_i^{(k)}+a\bigr),$$

how much topic $k$ likes word $v$, times how much statement $i$ already uses topic $k$. After 300 sweeps,
$\hat\phi$ and $\hat\theta$ are read off the counts. It sees the 149 training statements only. It takes about
a minute.

**The thermometer.** Every ten sweeps the sampler records the log-likelihood of the observed words under the
current bars and shares, $\sum_i \sum_{\text{tokens } v \text{ in } i} \log \sum_k \hat\theta_{ik}\hat\phi_{kv}$. It is a
monitor, not an objective: Gibbs maximises nothing. Random tags make every word improbable, so it starts low;
as the tags sort themselves it rises; once the chain has settled it is flat, and that is when the counts are read.

In [ ]:
t0 = time.time()
lda = tx.fit_lda_gibbs(train, K=GATE["lda_K"], sweeps=GATE["lda_sweeps"])
print(f"K = {lda.K}, a = {lda.a}, b = {lda.b}, {lda.sweeps} sweeps, seed {lda.seed}: "
      f"{len(lda.vocabulary)} words, {lda.tokens:,} tokens, {time.time() - t0:.0f} s")
for k in range(lda.K):
    print(f"topic {k + 1}: " + ", ".join(lda.top_words(k, 10)))

shares = lda.yearly_shares()
sweeps_at = [s for s in range(1, lda.sweeps + 1) if s % 10 == 0 or s == lda.sweeps]   # when the trace was taken
print(f"thermometer: {lda.loglik_trace[0]:,.0f} at sweep {sweeps_at[0]} -> {lda.loglik_trace[-1]:,.0f} at sweep {lda.sweeps}")
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.7]})
ax0.plot(sweeps_at, lda.loglik_trace, color=TEAL)
ax0.set(xlabel="sweep", ylabel="log-likelihood of the words", title="the thermometer")
ax1.stackplot(shares.index.astype(int), shares.T.values, labels=shares.columns,
              colors=[TEAL, "#B38A38", RED, MUTED], alpha=0.6)
ax1.set_xlim(1994, 2014); ax1.set_ylim(0, 1); ax1.set_ylabel("mean topic share")
ax1.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
plt.tight_layout(); plt.show()
print("Read the top words, then name each topic before looking at the plot. Then ask a neighbour to name")
print("them without seeing yours. Where you disagree, the name is a choice, not a finding.")
print("Then refit: tx.fit_lda_gibbs(train, K=4, seed=7) and tx.fit_lda_gibbs(train, K=6). Which topics")
print("survive the new seed, which split when K grows, and do the topic numbers stay put?")

ok = (lda.top_words(0, 8) == REC["text"]["lda"]["top_words"][0]
      and [lda.top_words(k, 8) for k in range(lda.K)] == REC["text"]["lda"]["top_words"]
      and len(lda.vocabulary) == GATE["lda_vocabulary"] and lda.tokens == GATE["lda_tokens"]
      and max(lda.dates) <= GATE["corpus_training_end"])
gate("A4", ok, f"{lda.K} topics fitted on {len(lda.dates)} statements, all dated on or before "
               f"{GATE['corpus_training_end']}; the top words reproduce the recorded run")

## A5b. LDA by hand: the story, one redraw, the counts

A5 fitted four topics to 149 statements. Here the method works on A4b's three documents, small enough to follow
one word. First **the story** LDA assumes, with the two topic bars of 9.A1's "The Assumption" and $d_3$'s shares
$\theta_3=(0.1,\,0.9)$: for each slot, roll the document's die for a topic, then that topic's die for a word.
Then **the fit**, which runs the story backwards, as in "How LDA Finds the Topics": give every word a random
colour, forget one word's colour, and redraw it with odds

$$\text{odds}(k)=\bigl(n_i^{(k)}+a\bigr)\times\frac{n_k^{(v)}+b}{n_k^{(\cdot)}+Vb},$$

how much its statement $i$ uses colour $k$, times how much colour $k$ uses this word $v$ elsewhere, every count
taken without the word itself; $a=0.1$, $b=0.01$ and $V=8$. Sweep the corpus many times, then count.

In [ ]:
story = cx.lda_story()
print("the story, one document of five words:  " + "  ".join(f"{w} (topic {z})" for z, w in story))
print("Each slot rolls the document's die for a topic, then that topic's die for a word; order plays no part.\n")

start = cx.random_colours()
for name, doc, tags in zip(("d1", "d2", "d3"), cx.TOY_DOCS, start):
    print(f"step 1, {name}:  " + "  ".join(f"{w}({k})" for w, k in zip(doc.split(), tags)))
print("d3 is coloured as 9.A1 colours it; d1 and d2 at random, seed 42.\n")

print("steps 2-3 for each word of d3 in turn, from that colouring (counts without the word itself):")
print(f"{'word':<11}{'colour':>7}{'statement uses 1, 2':>21}{'colour 1, 2 uses word':>24}{'P(1)':>8}{'P(2)':>7}")
for pos, w in enumerate(cx.TOY_DOCS[2].split()):
    r = cx.redraw_odds(start, doc=2, pos=pos)
    print(f"{w:<11}{r['old']:>7}{r['statement_counts'][0]:>12.0f}{r['statement_counts'][1]:>9.0f}"
          f"{r['word_counts'][0]:>14.0f}{r['word_counts'][1]:>10.0f}{r['p'][0]:>8.3f}{r['p'][1]:>7.3f}")
hike = cx.redraw_odds(start, doc=2, pos=1)
lean, other, win = (1 + int(np.argmax(hike[key])) for key in ("statement", "word_counts", "p"))
print(f"\n'hike': its statement leans to colour {lean} ({hike['statement'][0]:.1f} against {hike['statement'][1]:.1f}, "
      f"a included), but its one other occurrence,")
print(f"in d2, has colour {other}, so colour {win} wins with probability {hike['p'].max():.2f}. 'respond' occurs nowhere "
      "else: a coin toss.")

run = cx.toy_gibbs(start, sweeps=200)
print("\nstep 4, sweep after sweep: d3's five colours (fed hike respond inflation inflation)")
for sweep in (1, 2, 3, 4, 5, 50, 100, 200):
    print(f"   after sweep {sweep:>3}: {run['trace'][sweep - 1]}")
print("Single colours keep flickering, as 9.A1 says. Then count, after sweep 200: each colour's bar, and the shares")
print(pd.DataFrame(run["phi"], index=["colour 1", "colour 2"], columns=cx.TOY_VOCAB)
      .to_string(float_format=lambda x: f"{x:.2f}"))
print("shares: " + "   ".join(f"d{i + 1} (" + ", ".join(f"{x:.2f}" for x in row) + ")" for i, row in enumerate(run["theta"])))

toy = tx.toy_lda()                               # A5's sampler on the same three documents: three seeds
print("\nA5's sampler on the same three documents, from random colourings, 200 sweeps under each of three seeds:")
for r in toy:
    print(f"   seed {r['seed']:>2}: topic 1 " + ", ".join(r["top_words"][0]) + "  |  topic 2 "
          + ", ".join(r["top_words"][1]) + f"    d3's shares ({r['theta'][2][0]:.2f}, {r['theta'][2][1]:.2f})")
col = list(cx.TOY_VOCAB)
here = np.argmax(run["phi"][:, col.index("hike")]) == np.argmax(run["phi"][:, col.index("inflation")])
there = [any({"hike", "inflation"} <= set(t) for t in r["top_words"]) for r in toy]
print("hike and inflation in one colour: " + ("yes" if here else "no") + " in the run from 9.A1's colouring; "
      + ", ".join(f"{'yes' if t else 'no'} under seed {r['seed']}" for r, t in zip(toy, there)) + " in A5's.")
print("Thirteen words give the sampler little to settle on: where it ends depends on where it starts, and one")
print("division can come back numbered either way. On A5's 149 statements the thermometer flattens before the")
print("counts are read. On any corpus, refit under another seed and read a topic's words before naming it.")

rec_toy = REC["text"]["lda_toy"]["runs"]
keys = ("seed", "K", "sweeps", "a", "b", "top_words", "phi", "theta")
gate("A5b", [{k: r[k] for k in keys} for r in toy] == [{k: r[k] for k in keys} for r in rec_toy],
     "the three-document sampler reproduces the recorded run under all three seeds: top words, bars and shares")

# Block B — Vectors, and a measurement that fails

A TF-IDF column knows nothing about any other column: "raise" and "increase" are as unrelated as "raise" and
"unemployment". **Word vectors** fix that by reading a word's company.

## B1. Word vectors: co-occurrence, PPMI, SVD

Three steps, all on the training window:

1. **Co-occurrence.** Count how often each pair of words appears within ±5 words of one another.
2. **PPMI.** Replace the count by $\max\{0, \log \frac{p(i,j)}{p(i)p(j)}\}$ — how much more often the pair
   occurs than chance would give. Zero for pairs no more frequent than chance, which is most of them.
3. **SVD.** Keep the first 100 singular directions, then normalise every row to unit length, so a dot product
   is a cosine and nothing depends on how often a word happens to appear.

`np.linalg.svd` on a dense matrix: deterministic, no seed, no randomised solver.

In [ ]:
emb = tx.fit_embeddings(train, window=GATE["embedding_window"], min_count=GATE["embedding_min_count"],
                        dim=GATE["embedding_dim"])
print(f"{len(emb.vocabulary)} words survive min_count = {emb.min_count} in {emb.n_fit} statements; "
      f"vectors are {emb.dim}-dimensional")
print(f"PPMI matrix is {100 * emb.ppmi_density:.1f}% non-zero; first singular values "
      f"{np.round(emb.singular_values[:6], 1)}\n")

for word in ("inflation", "employment", "committee", "accommodation"):
    print(f"{word:<15} " + ", ".join(f"{n} {v:.2f}" for n, v in tx.neighbors(emb, word, k=6)))

print("\ntwo analogies, b - a + c; in the first, all three inputs have vectors:")
print("   increase : increased :: moderate : " +
      ", ".join(f"{w} {v:.2f}" for w, v in tx.analogy(emb, "increase", "increased", "moderate", k=3)))
print(f"   a miss: the answer, 'moderated', occurs {tx.word_frequencies(train)['moderated']} times in the window, "
      f"below min_count = {emb.min_count}, so it has no vector")
print("in the second, an input has none:")
try:
    tx.analogy(emb, "raise", "raised", "lower")
except KeyError as exc:
    print(f"   raise : raised :: lower : ?  —  {exc}")
print("A word below min_count has no vector at all. That is a property of the method, not a defect of")
print("this run, and it is why the analogy table in the recorded run carries nulls rather than guesses.")

ok = (emb.n_fit == GATE["embedding_n_fit"] and len(emb.vocabulary) == GATE["embedding_vocabulary"]
      and emb.min_count == GATE["embedding_min_count"] and emb.dim == GATE["embedding_dim"])
gate("B1", ok, f"co-occurrence, PPMI and SVD from {emb.n_fit} training statements only; "
               f"{len(emb.vocabulary)} words at min_count {emb.min_count}")

## B1b. What word2vec does instead

B1 counted co-occurrences and compressed them. Word2vec, in 9.A2's version (skip-gram with negative sampling),
learns the vectors by classification. **Step 1** makes labelled rows out of unlabelled text: each (centre,
context) pair inside the window is real, label 1; for each real pair, $K$ noise words drawn from the vocabulary
with probability $\propto f(w)^{3/4}$ give pairs labelled 0. **Step 2** fits a logit,
$P(\text{real})=\sigma(\tilde{\boldsymbol u}_c^{\top}\boldsymbol u_w)$, whose only coefficients are the word
vectors. The two routes are close relatives: skip-gram with $K$ noise words implicitly factorises the matrix of
pointwise mutual information shifted by $\log K$ (Levy and Goldberg 2014). B1 builds that matrix explicitly,
unshifted and clipped at zero (PPMI), and compresses it with an SVD.

In [ ]:
noise = tx.noise_draws(emb)                      # B1's vocabulary and its counts on the window; seed 42
print(f"{noise['vocabulary_size']} words, {noise['tokens']:,} tokens in the training window\n")
print(f"{'':<16}{'word':<10}{'count':>7}{'f(w), %':>10}{'P_noise, %':>12}{'ratio':>8}")
for label, r in noise["rows"].items():
    print(f"{label.replace('_', ' '):<16}{r['word']:<10}{r['count']:>7,}{100 * r['f']:>10.3f}"
          f"{100 * r['p_noise']:>12.3f}{r['ratio']:>8.2f}")
print(f"the ten commonest words: {100 * noise['top10_share_f']:.1f}% of the tokens, "
      f"{100 * noise['top10_share_noise']:.1f}% of the noise draws\n")
print(f"'{noise['sentence']}': centre '{noise['centre']}', window {noise['window']}, K = {noise['k']}")
n_rows = 0
for pair in noise["pairs"]:
    print(f"   ({noise['centre']}, {pair['context']}) 1     "
          + "   ".join(f"({noise['centre']}, {w}) 0" for w in pair["noise"]))
    n_rows += 1 + len(pair["noise"])
print(f"{n_rows} rows from one centre word: 2 x {noise['window']} real pairs and 2 x {noise['window']} x "
      f"{noise['k']} noise pairs")

print("\nstep 2, the logit: " + ", ".join(f"sigma({x}) = {cx.sigmoid(x):.2f}" for x in (0, 4, -4)))
st = cx.sgns_step()
print(f"one step on 9.A2's two-dimensional vectors, eta = {cx.SGNS_ETA}:")
print(f"   real pair (raised, rates): x+ = {st['x_plus']:.1f}, sigma(x+) = {st['s_plus']:.2f};  noise pair: "
      f"x- = {st['x_minus']:.1f}, sigma(x-) = {st['s_minus']:.2f}")
for name, label in (("u_raised", "u_raised"), ("u_rates", "u~_rates"), ("u_noise", "u~_noise")):
    print(f"   {label:<9} before ({', '.join(f'{v:.3f}' for v in st['before'][name])}), "
          f"after ({', '.join(f'{v:.3f}' for v in st[name])})")
print(f"   x+ {st['x_plus']:.2f} -> {st['x_plus_after']:.2f}: the real context pulled closer;  "
      f"x- {st['x_minus']:.2f} -> {st['x_minus_after']:.2f}: the noise pushed away")
print(f"   log sigma(x+) + log sigma(-x-): {st['objective_before']:.3f} before the step, "
      f"{st['objective_after']:.3f} after")

rec_noise = {k: v for k, v in REC["text"]["noise"].items() if k != "_recorded"}
after = all(cx.printed_as(st[k][i], v[i]) for k, v in cx.SLIDE_SGNS_AFTER.items() for i in range(2))
gate("B1b", noise == rec_noise and after,
     "the noise draws and their rows reproduce the recorded run; one step moves the three vectors to the "
     "points 9.A2 plots, at its three decimals")

## B1c. Window and dimension

Two choices 9.A2 leaves to the researcher: the window $\Delta$ and the length $d$ of the vectors. B1 used
$\Delta=5$ and $d=100$. The cell refits on the same 149 statements at $\Delta=2$ and $\Delta=10$, then at
$d=25$ and $d=300$, and prints nearest neighbours. 9.A2's rule of thumb, a small window finds syntax and a large
one the topic, is tried on two words.

In [ ]:
fits = {(window, dim): tx.fit_embeddings(train, window=window, min_count=GATE["embedding_min_count"], dim=dim)
        for window, dim in ((2, 100), (10, 100), (5, 25), (5, 300))}
for word in cx.WINDOW_WORDS:
    print(word)
    for window in (2, 10):
        print(f"   window {window:>2}: "
              + ", ".join(f"{n} {v:.2f}" for n, v in tx.neighbors(fits[(window, 100)], word, k=6)))
    common = cx.overlap(tx.neighbors(fits[(2, 100)], word, k=10), tx.neighbors(fits[(10, 100)], word, k=10))
    print(f"   {common} of the ten nearest words are the same under both windows")
ng = cx.ngram_counts(train)
bg = ng["bigram_counts"]
print("\n'balance' in the window: " + ", ".join(f"{a} {b} {bg[(a, b)]}" for a, b in
                                            (("on", "balance"), ("in", "balance"), ("balance", "of"), ("balance", "sheet"))))
print("Window 2 reads only the words beside 'balance', most often 'on balance', and none of its six neighbours is")
print("about the balance sheet; window 10 reads the passage and finds the subject, the balance sheet: its 'size' and")
print("'composition'. On 'balance' the rule of thumb holds.")
print(f"On 'discount' it does not: {bg[('discount', 'rate')]} of its {ng['unigram_counts']['discount']} uses in the window are "
      "'discount rate', and at both windows")
print("its neighbours come from the discount-rate announcements (board and district; rate, directors and banks). A")
print("word that lives in one phrase is used one way, so a small window has no syntax to find. In this corpus a")
print("discount is the discount rate. 9.A2 warns that vectors trained on general text put it near shop markdowns;")
print("C0, below, shows a general-purpose model barely telling the two apart. Train on the corpus you measure.\n")
for word in cx.DIMENSION_WORDS:
    print(word)
    for dim in (25, 300):
        near = tx.neighbors(fits[(5, dim)], word, k=6)
        print(f"   d = {dim:>3}: " + ", ".join(f"{n} {v:.2f}" for n, v in near)
              + f"   (sixth nearest {near[-1][1]:.2f})")
low = min(tx.neighbors(fits[(5, 25)], word, k=6)[-1][1] for word in cx.DIMENSION_WORDS)
print(f"\nAt d = 25, below the usual 100-300, even the sixth nearest word scores {low:.2f} or more: a short vector")
print(f"squeezes many words together. At d = 300, for {len(emb.vocabulary)} words, the vectors keep more of this corpus's")
print("accidents and the scores fall. The neighbours move with d, so d is reported with them.")

## B1d. Directions: 9.A2's nine points, then B1's vectors

9.A2's "The Geometry" and "Analogies, Worked" place nine words by hand on three named axes (stance, cycle,
topic), so that the arithmetic can be read by eye. The cell redoes it from those points: the clusters, the two
parallel moves and their cosine, the analogy hawkish − hike + cut, and why distance reads the lengths that
cosine ignores. Then it asks B1's fitted vectors the slides' question, with the two pairs whose words both have
a vector there: does the move from a tightening word to an easing word point the same way?

In [ ]:
P = {w: np.array(v) for w, v in cx.HAND_SET.items()}
cosine = lambda u, v: float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
dist = lambda u, v: float(np.linalg.norm(u - v))
fmt = lambda v: "(" + ", ".join(f"{x:.1f}" for x in v) + ")"
print(f"{'':<14}" + "".join(f"{a:>8}" for a in cx.HAND_AXES))
for w, v in P.items():
    print(f"{w:<14}" + "".join(f"{x:>8.1f}" for x in v))
within = lambda g: np.mean([dist(P[a], P[b]) for a, b in itertools.combinations(g, 2)])
between = lambda g, h: np.mean([dist(P[a], P[b]) for a in g for b in h])
print("\nmean distance inside a cluster:  " + ", ".join(f"{n} {within(g):.2f}" for n, g in cx.CLUSTERS.items()))
print("and between two clusters:        " + ", ".join(f"{m} to {n} {between(cx.CLUSTERS[m], cx.CLUSTERS[n]):.2f}"
                                                  for m, n in itertools.combinations(cx.CLUSTERS, 2)))
top = max(P, key=lambda w: P[w][2])
print(f"coupling: '{top}' is {P[top][0]:.1f} on stance and {P[top][2]:.1f} on topic, the highest topic score of the nine")

hc, hd = P["hike"] - P["cut"], P["hawkish"] - P["dovish"]
query = P["hawkish"] - P["hike"] + P["cut"]
d_near, w_near = min((dist(query, P[w]), w) for w in P if w not in ("hawkish", "hike", "cut"))
print(f"\nhike - cut = {fmt(hc)}, hawkish - dovish = {fmt(hd)}: cosine {cosine(hc, hd):.4f}")
print(f"hawkish - hike + cut = {fmt(query)}: the nearest point is '{w_near}', {d_near:.2f} away")
print(f"distance reads lengths: hike and 2 x hike have cosine {cosine(P['hike'], 2 * P['hike']):.2f} "
      f"yet lie {dist(P['hike'], 2 * P['hike']):.2f} apart")

(a, b), (c, d) = cx.STANCE_PAIRS
moves = cosine(emb[a] - emb[b], emb[c] - emb[d])
answers = tx.analogy(emb, a, b, c, k=3)
print(f"\nB1's vectors, {emb.dim} unnamed axes: {a} - {b} against {c} - {d}, cosine {moves:.2f}")
print(f"   {a} : {b} :: {c} : " + ", ".join(f"{w} {v:.2f}" for w, v in answers))
for w in (a, b):
    print(f"   nearest to '{w}': " + ", ".join(f"{n} {v:.2f}" for n, v in tx.neighbors(emb, w, k=4)))
after = collections.Counter({y: n for (x, y), n in cx.ngram_counts(train)["bigram_counts"].items() if x == b})
print(f"   the words after '{b}' in the window: " + ", ".join(f"{y} {n}" for y, n in after.most_common(4)))
length = np.linalg.norm(emb.vectors, axis=1)
print(f"   every vector has length 1 (within {np.abs(length - 1).max():.0e}), so here distance ranks neighbours as cosine does")
print("\nThe nine points were placed so that the moves are parallel. B1's vectors were fitted, and their two moves")
print(f"from a tightening word to an easing word barely align: cosine {moves:.2f}, not {cosine(hc, hd):.4f}; the analogy "
      + ("does not return" if d not in [w for w, _ in answers] else "returns") + f" '{d}'.")
print(f"'{a}' sits with the words of the decision sentence; '{b}' is a decision in '{b} its' and a comparative")
print("elsewhere, and one vector per word averages the two uses (9.A2, 'Four Limits of One Vector per Word').")
print("B4 meets the same fact from the other side: an axis built from these vectors reads vocabulary.")

S = cx.SLIDE_GEOMETRY
gate("B1d", (fmt(hc), fmt(hd), f"{cosine(hc, hd):.4f}", fmt(query), w_near, f"{d_near:.2f}")
     == (S["hike_cut"], S["hawk_dove"], S["cosine"], S["query"], "dovish", S["distance"]),
     "9.A2's two moves, their cosine, and the analogy's point and its distance from 'dovish', at its printed precision")

## B2. Two measurements built on those vectors

**A hawk–dove axis.** Average the vectors of the hawkish seed words (raise, increase, increased, increasing,
tightening, firming), average the dovish ones (lower, easing, eased, accommodative, accommodation), subtract, and
normalise. Seeds seen fewer than ten times have no vector and drop out: five hawkish and four dovish remain. A statement is scored by the cosine between that axis and its TF-IDF-weighted mean word vector, so
template text carries little weight. Positive is meant to read hawkish.

**A novelty index.** $1 - \cos(x_t, \bar x_{t-8:t-1})$: how far this meeting's language sits from what the
Committee has been saying lately. It is not a stance proxy and is not offered as one; it answers a different
question, and it is here so that the two can be told apart.

In [ ]:
hawkdove = tx.signal_embedding_hawkdove(statements, emb, model)
dates = [s.date for s in statements]
print("hawk seeds inside the vocabulary:", {w: hawkdove["hawk_counts"][w] for w in hawkdove["hawk_used"]})
print("dove seeds inside the vocabulary:", {w: hawkdove["dove_counts"][w] for w in hawkdove["dove_used"]})
print("(counts are occurrences in the training window; 'raise' and 'firming' behave very differently"
      " from 'accommodation')\n")
print("the eight most novel statements:", ", ".join(f"{d} {v:.2f}" for d, v in hawkdove["most_novel"][:8]))

x = np.arange(len(dates))
fig, ax = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True)
labels = tx.cycle_labels(dates)
for a in ax:
    for i in range(len(dates) - 1):
        if labels[i] != 0:
            a.axvspan(i, i + 1, color=TEAL if labels[i] > 0 else RED, alpha=0.12, lw=0)
ax[0].plot(x, hawkdove["score"], color=MUTED, lw=0.9)
ax[0].axhline(0, color="0.7", lw=0.6)
ax[0].set(ylabel="hawk–dove axis", title="shaded: hiking cycles (teal) and cutting cycles (red)")
ax[1].plot(x, hawkdove["novelty"], color=MUTED, lw=0.9)
ax[1].set(ylabel="novelty index", xlabel="meeting")
ticks = [i for i in range(0, len(dates), 30)]
ax[1].set_xticks(ticks); ax[1].set_xticklabels([dates[i][:7] for i in ticks])
plt.tight_layout(); plt.show()

## B3. Validation: against what the Committee then did

A stance read at meeting $t$ is a claim about where policy goes next, so the thing to line it up with is not
the label of meeting $t$ but the **subsequent policy path**: the mean cycle label over the next three meetings,
$+1$ inside a hiking band, $-1$ inside a cutting band, $0$ between cycles. The cycles are the Fed's own: each
runs from the first to the last move of a run of hikes or of cuts (`tx.HIKING`, `tx.CUTTING`).

Two verdicts are reported for each signal. The **sign test** — every hiking band positive and every cutting
band negative — because the units of these signals are not comparable and a threshold would be arbitrary. And
**Spearman's rank correlation with the subsequent path**, which says how much of the ordering survives.

In [ ]:
regex = tx.signal_decision_sentence(statements, scope="statement")
checks = [tx.validate_signal(dates, hawkdove["score"], "embedding hawk-dove",
                             horizon=GATE["validation_horizon"]),
          tx.validate_signal(dates, regex["score"], "decision-sentence regex",
                             horizon=GATE["validation_horizon"])]
print(f"scored inside the training window only (end = {checks[0]['window_end']}), "
      f"{checks[0]['n_scored']} statements each\n")
print(f"{'signal':<26}{'hiking mean':>13}{'cutting mean':>14}{'gap':>9}"
      f"{'rank corr. w/ path':>21}{'verdict':>10}")
for v in checks:
    print(f"{v['signal']:<26}{v['hiking_mean']:>13.3f}{v['cutting_mean']:>14.3f}{v['gap']:>9.3f}"
          f"{v['rank_correlation_with_subsequent_path']:>21.3f}"
          f"{'PASSES' if v['passes'] else 'FAILS':>10}")
print("\nper cycle — where each signal works and where it does not:")
cycles = (tx.validation_table(checks).pivot(index="cycle", columns="signal", values="mean")
          .dropna(how="all")[[v["signal"] for v in checks]])
print(cycles.to_string())
print("The last four cycles begin after 2014 and sit outside the training window, so neither signal")
print("is scored on them here.")

tol = GATE["validation_tolerance"]
recorded = [REC["text"]["validation"][f"{v['signal']} | training window"] for v in checks]
ok = all(abs(v["rank_correlation_with_subsequent_path"]
             - r["rank_correlation_with_subsequent_path"]) < tol
         and v["passes"] == r["passes"] and v["n_scored"] == r["n_scored"] for v, r in zip(checks, recorded))
gate("B2", ok, f"both signals reproduce the recorded run to {tol:g}: rank correlations "
               f"{checks[0]['rank_correlation_with_subsequent_path']:+.3f} and "
               f"{checks[1]['rank_correlation_with_subsequent_path']:+.3f}")

In [ ]:
# A common repair: centre the document vectors (subtract their mean) before taking a cosine. Does centring rescue this score? No.
centred = tx.embedding_centred_check(train, emb, model)
print(f"mean cosine between two statements' document vectors: "
      f"{centred['mean_pairwise_cosine_of_document_vectors']:.2f}  (a large common part)")
for row in centred["rows"]:
    print(f"  {row['variant']:<32} rank correlation with the policy path {row['rank_correlation']:+.3f}   "
          f"{'passes' if row['passes'] else 'fails'}")
print("recorded:", [(r["variant"], r["rank_correlation"]) for r in REC["text"]["embedding_centred"]["rows"]])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.1))
for a, v in zip(ax, checks):
    rows = [r for r in v["per_cycle"] if r["covered"]]
    colour = [TEAL if r["kind"] == "hiking" else RED for r in rows]
    a.bar(range(len(rows)), [r["mean"] for r in rows], color=colour)
    a.axhline(0, color="k", lw=0.7)
    a.set_xticks(range(len(rows)))
    a.set_xticklabels([r["start"] for r in rows], rotation=45, ha="right", fontsize=7)
    a.set_title(f"{v['signal']}\n{'passes' if v['passes'] else 'FAILS'} the sign test", fontsize=9)
ax[0].set_ylabel("band mean")
plt.tight_layout(); plt.show()
print("A hiking band (teal) should stand above zero and a cutting band (red) below it.")
worst = checks[0]["worst_hiking_band"]
print(f"For the hawk–dove axis, the {worst['start']} to {worst['end']} hiking cycle — {worst['n']} meetings"
      f" — has mean {worst['mean']:+.4f}:")
print("the most negative band anywhere in the training window belongs to a two-year run of rate increases.")

## B4. This is the result, not a defect

The embedding hawk–dove score reads the 2004–06 tightening cycle as the most dovish stretch in the sample.
Nothing is broken. The axis measures **vocabulary**, and in 2004–06 the Committee's vocabulary was dovish
while its decision was not: it was *removing accommodation*, at a *measured* pace, and said so at every
meeting through 2005. A mean document vector built from the whole statement is pulled toward "accommodation" — a dove seed
word — precisely because the Committee was taking accommodation away.

The cell below counts the words. Then it prints the sentence the regex reads.

In [ ]:
band = [i for i, d in enumerate(dates) if "2004-06" <= d[:7] < "2006-07"]
rest = [i for i, d in enumerate(dates) if i not in band and d <= tx.TRAIN_END]
def per_statement(idx, group):
    return np.mean([sum(tx.BASELINE.run(statements[i].text).count(w) for w in group) for i in idx])

print(f"{'words':<46}{'2004-06 cycle':>15}{'rest of window':>17}")
for group in (["accommodation"], ["accommodative"], ["measured"], ["raise", "raised", "raising"],
              ["increase", "increased", "increasing"]):
    print(f"{', '.join(group):<46}{per_statement(band, group):>15.2f}{per_statement(rest, group):>17.2f}")
print("\nper statement. 'accommodation' nearly doubles; 'measured' goes from rare to almost every meeting;")
print("and the word that actually names the decision, 'raise', goes from 0.08 to 1.00 — once per statement.\n")

for i in (band[0], band[len(band) // 2], band[-1]):
    print(f"{dates[i]}  hawk {regex['hawk_matches'][i]:.0f} dove {regex['dove_matches'][i]:.0f}  |  "
          f"{tx.decision_sentence(statements[i].text)}")
print("\nThe regex is ten lines: a decision verb within six words of the policy instrument, counted over the whole")
print("statement. It passes because it counts decisions, not vocabulary: outside sentences like the three above,")
print("the statements seldom put such a verb next to the funds rate. B4b narrows it to the decision sentence alone.")
print("Identification precedes measurement: deciding what sentence carries the quantity comes first, and no")
print("amount of representation learning substitutes for it.")

## B4b. The same rule, narrowed to one sentence

B3 counted the two patterns over the whole statement. `tx.decision_sentence` picks out the sentence that
announces the decision, the first that names the policy instrument and carries a decision verb (B4 printed three),
and the cell runs the same two patterns over that sentence alone, then validates the result exactly as B3 did.
What a measure reads is chosen before anything is computed.

In [ ]:
narrow = tx.signal_decision_sentence(statements, scope="decision_sentence")
check = tx.validate_signal(dates, narrow["score"], "decision-sentence regex, narrowed",
                           horizon=GATE["validation_horizon"])
window = [i for i, d in enumerate(dates) if d <= tx.TRAIN_END]
fires = lambda sig: 100 * float(np.mean([sig["score"][i] != 0 for i in window]))
print(f"{'signal':<36}{'hiking mean':>13}{'cutting mean':>14}{'rank corr. w/ path':>21}{'verdict':>9}{'fires on':>10}")
for sig, v in ((regex, checks[1]), (narrow, check)):
    print(f"{v['signal']:<36}{v['hiking_mean']:>13.3f}{v['cutting_mean']:>14.3f}"
          f"{v['rank_correlation_with_subsequent_path']:>21.3f}{'PASSES' if v['passes'] else 'FAILS':>9}{fires(sig):>9.1f}%")
print(f"('fires on': the share of the {len(window)} window statements with a non-zero score)")
print(f"\nhow the sentence was found, all {len(statements)} statements: "
      + ", ".join(f"{k} {n}" for k, n in narrow["sentence_source"].items()))
differ = [i for i in window if regex["score"][i] != narrow["score"][i]]
print(f"the two scores differ on {len(differ)} window statements:")
for i in differ:
    print(f"   {dates[i]}  whole statement: hawk {regex['hawk_matches'][i]:.0f} dove {regex['dove_matches'][i]:.0f}"
          f"   decision sentence: hawk {narrow['hawk_matches'][i]:.0f} dove {narrow['dove_matches'][i]:.0f}")
print("\nThe matches the narrowing drops sit in other sentences. In 1999 a second sentence restates the increase")
print("('Today's increase in the federal funds rate'); in 2006 a dissent prefers an increase while the Committee")
print("holds; in 1998 a paragraph explains why 'a slightly lower federal funds rate' fits. Both versions pass, and")
print("the narrowed one ranks the path better.")

rec = REC["text"]["validation"]["decision-sentence regex, narrowed | training window"]
tol = GATE["validation_tolerance"]
gate("B4b", abs(check["rank_correlation_with_subsequent_path"] - rec["rank_correlation_with_subsequent_path"]) < tol
     and check["passes"] == rec["passes"] and check["n_scored"] == rec["n_scored"],
     f"the narrowed rule reproduces the recorded run to {tol:g}: rank correlation "
     f"{check['rank_correlation_with_subsequent_path']:+.3f}, {'passes' if check['passes'] else 'fails'}")

# Block C — Inside the small transformer

## C0. One word, two sentences

A word vector from B1 gives *rate* one point, whatever the sentence. An encoder such as BERT gives it a new
vector in every sentence it reads, because each word's vector is rewritten from the words around it: block C
opens that machinery. The recorded run fed multilingual BERT three sentences per probe: an anchor, a sentence
using the word in the same sense, and one using it in another sense or negated. Each cosine compares the word's
last-layer vector in the anchor with its vector in one of the others; a static vector would score exactly 1
every time. Nothing here calls a model: the numbers are read from `recorded_deck12_examples.json`, after its
hash is checked.

In [ ]:
record = cx.load_record()
meta = record["contextual"]
print(f"{meta['model']}: {meta['layers']} layers, d = {meta['hidden']}, {meta['heads']} heads, "
      f"{meta['parameters']:,} parameters")
print(f"the vector: {meta['vector']}")
print(f"against a static vector: {meta['static_baseline']}\n")
probes = cx.contextual_rows(record)
print("each probe: the anchor sentence, then the cosine of the word's vector there with its vector in each other sentence\n")
for r in probes:
    cut = "" if len(r["pieces"]) == 1 else f", cut into {' | '.join(r['pieces'])}"
    print(f"'{r['target']}'{cut}: {r['contrast']}")
    print(f"   anchor            {r['anchor']}")
    print(f"   same     {r['cos_same']:.3f}    {r['same']}")
    print(f"   other    {r['cos_other']:.3f}    {r['other']}")
print("\nsame sense minus other sense: " + ", ".join(f"{r['target']} {r['cos_same'] - r['cos_other']:+.3f}"
                                                   for r in probes))
print("The two senses of 'rate', and of 率, separate clearly. Negation hardly moves a vector: 'not sufficient' sits")
print("almost where 'sufficient' does, and 'not slowing' sits closer to the anchor than 'clearly slowing'. A shop's")
print("discount sits nearly as close as the discount rate. A contextual vector carries the sentence's wording; it")
print("does not by that fact carry its logic, and a general corpus is not the Committee's.")

## C1. The arithmetic, by hand

Attention is a weighted average whose weights are computed from the data. For a query at position $i$:
score every position, $s_{ij} = \langle q_i, k_j\rangle / \sqrt{d_h}$; soften the scores into weights,
$a_{ij} = e^{s_{ij}} / \sum_k e^{s_{ik}}$; average the values with those weights. That is the whole mechanism.

Chapter 13's example box stipulates nine scores for the sentence *"The central bank raised interest rates
amid rising inflation"*, with "interest" as the query. **The chapter says in terms that the scores are set
for exposition and are not the output of any model**, so what is reproduced here is the arithmetic, not the
scores.

In [ ]:
book = at.book_attention()
print(f"query: position {book['query_position']}, '{book['query_token']}'  —  "
      f"{len(book['tokens'])} tokens\n")
print(f"{'position':<10}{'token':<12}{'score':>8}{'exp(score)':>13}{'weight':>10}")
for i, t in enumerate(book["tokens"], start=1):
    print(f"{i:<10}{t:<12}{book['scores'][t]:>8.1f}{book['exponentials'][t]:>13.4f}"
          f"{book['weights'][t]:>10.4f}")
print(f"{'':<22}{'sum':>8}{book['sum_of_exponentials']:>13.4f}{book['weights_sum']:>10.4f}")
print(f"\nentropy of the row: {book['entropy']:.4f} nats, against ln {len(book['tokens'])} = "
      f"{np.log(len(book['tokens'])):.4f} for a flat row\n")
print(f"{'quantity':<28}{'chapter prints':>16}{'computed here':>16}   agrees")
for row in book["comparison"]:
    print(f"{row['quantity']:<28}{row['printed']:>16}{row['computed_to_printed_precision']:>16}   "
          f"{'yes' if row['agrees'] else 'NO'}")
print("\n" + book["rounding_note"])
print("\n" + book["stipulation"])

ok = (book["all_agree"] and abs(book["sum_of_exponentials"] - GATE["book_attention_sum_of_exponentials"]) < 5e-4
      and len(book["comparison"]) == GATE["book_attention_digits_checked"])
gate("C1", ok, f"all {len(book['comparison'])} printed figures reproduce at the chapter's own precision; "
               f"the exponentials sum to {book['sum_of_exponentials']:.4f}")

In [ ]:
five = at.lab7b_five_score_variant()
nine = {row["quantity"]: row["computed_to_printed_precision"] for row in book["comparison"]}
print("softmax over the five named scores only, the four positions that score 0 left out")
print("   five-token softmax :", {k: f"{v:.2f}" for k, v in five["weights"].items()})
print("   nine-token softmax :", {k: nine[k] for k in five["weights"]})
print("  ", five["difference_from_the_nine_token_softmax"])
print("\nThe two are the same formula over different sets. A slide that shows both has to say which is which,")
print("because the four zero-scored positions carry 0.04 each and their absence lifts every other weight.")

## C1b. The same arithmetic with untrained weights, then masked

C1 started from stipulated scores. Here the scores are *computed*: give each of the nine tokens a random
16-number embedding, draw random $\mathbf W_Q, \mathbf W_K, \mathbf W_V$ (head width 8) and run the formula.
Nothing has been trained, so nothing tells *interest* which neighbours matter: its row comes out nearly flat,
close to $1/9$. The pattern in C1 is what training has to supply.

Then apply the **causal mask** a text generator needs: set $s_{ij} = -\infty$ for every $j > i$, so position $i$
cannot read the tokens after it. Those weights become exactly 0 and each row renormalises over $j \le i$.

In [ ]:
rng = np.random.default_rng(at.SEED)
X = rng.normal(size=(len(at.BOOK_TOKENS), 16)) / 4.0            # nine random embeddings, d = 16
W_Q, W_K, W_V, W_O = at.random_projections(16, 8, seed=at.SEED)  # untrained, head width d_h = 8
free = at.attention_from_embeddings(X, W_Q, W_K, W_V, W_O, causal=False)
masked = at.attention_from_embeddings(X, W_Q, W_K, W_V, W_O, causal=True)
q = at.BOOK_QUERY_POSITION - 1
book_masked = np.exp([at.BOOK_SCORES[t] for t in at.BOOK_TOKENS[:q + 1]])
book_masked = book_masked / book_masked.sum()

print(f"query '{at.BOOK_TOKENS[q]}' (position {q + 1})\n")
print(f"{'token':<12}{'untrained':>11}{'untrained, masked':>19}{'C1, masked':>12}")
for j, t in enumerate(at.BOOK_TOKENS):
    bm = f"{book_masked[j]:.3f}" if j <= q else "0"
    print(f"{t:<12}{free.weights[q, j]:>11.3f}{masked.weights[q, j]:>19.3f}{bm:>12}")
print(f"\nuntrained row: {free.weights[q].min():.3f} to {free.weights[q].max():.3f}, against 1/9 = {1/9:.3f}")
print(f"largest weight above the diagonal, masked: {np.triu(masked.weights, 1).max():.1e}")

rec = REC["attention"]["attention_by_hand"]
ok = (np.array_equal(np.round(free.weights[q], 4), np.array(rec["bidirectional"]["weights"]))
      and np.array_equal(np.round(masked.weights[q], 4), np.array(rec["causal"]["weights"]))
      and np.triu(masked.weights, 1).max() == 0.0)
gate("C1b", ok, "the untrained row and its masked version reproduce the recorded run at its four decimals; "
                "every weight above the diagonal is exactly 0")

## C1c. Many heads, no word order, and the triangle

Three properties of the mechanism, on C1b's nine random token vectors ($d=16$). **Heads**: split the width into
two heads of width 8, each with its own rows of $\mathbf W_Q,\mathbf W_K,\mathbf W_V$; the parameter count,
$4d^2$, does not depend on how many heads share it. **Order**: read the nine tokens backwards and, with no
position vectors and no mask, the outputs come back in reverse order, unchanged to the last digit: attention sees
tokens, not places. Add 9.A2's sinusoidal position vectors, or the causal mask, and that stops being so. **The
triangle**: the mask a decoder uses (9.A3) puts exactly 0 above the diagonal. An encoder such as BERT uses no
mask, so every token reads every other: the square.

In [ ]:
hot = cx.heads_order_triangle(X, n_heads=2)      # X: C1b's nine random token vectors
q = at.BOOK_QUERY_POSITION - 1
print(f"two heads of width {hot['head_dim']}, no mask: the row of '{at.BOOK_TOKENS[q]}'")
print(f"{'token':<12}{'head 1':>9}{'head 2':>9}")
for j, t in enumerate(at.BOOK_TOKENS):
    print(f"{t:<12}" + "".join(f"{hot['square']['weights'][h][q, j]:>9.3f}" for h in range(2)))
dev = float(np.abs(hot["square"]["weights"][:, q, :] - 1 / len(at.BOOK_TOKENS)).max())
print(f"Untrained, every weight in both rows is within {dev:.3f} of 1/{len(at.BOOK_TOKENS)}: random weights give no head a")
print("pattern; training gives each head its own.")
print(f"parameters in W_Q, W_K, W_V and W_O: {hot['parameters']:,} = 4 x {X.shape[1]}^2, for one head, two or four\n")
print("the nine tokens read backwards: the largest gap between the new outputs and the old ones, reversed")
for name, value in hot["backwards"].items():
    print(f"   {name:<26}{value:.1e}")
print("\nthe triangle, head 1: each row reads only the columns at or before it")
print(pd.DataFrame(hot["triangle"]["weights"][0].round(2), index=at.BOOK_TOKENS,
                   columns=[t[:5] for t in at.BOOK_TOKENS]).to_string())

bk = hot["backwards"]
gate("C1c", bk["no positions, square"] < 1e-12 and bk["positions added, square"] > 1e-6
     and bk["no positions, triangle"] > 1e-6 and hot["above_diagonal"] == 0.0,
     f"without positions or mask the outputs reverse with the tokens (gap {bk['no positions, square']:.0e}); "
     "positions or the mask break that; every weight above the diagonal is exactly 0")

## C2. Machine B, opened

Machine B is a character-level MiniGPT trained on these same 245 statements less every twentieth: $d_{\text{model}} = 128$, 4 heads,
3 layers, context 128 characters, a 78-character vocabulary, **631,424 parameters**. It is mirrored into
`l09_assets/minigpt/` under its hash manifest, and three things are proved before a single
weight is used: the five files match their recorded hashes; the class defined in `session05_attention.py`
produces exactly the state-dictionary keys and shapes the checkpoint carries; and the parameter count is the
one its training run reported. A checkpoint that loads with missing or unexpected keys is a different model wearing
the same filename.

One line was added to Machine B's model class and nothing else: `return_attn`, which hands back the attention
weights that every forward pass already computes and that the training script dropped on the floor.

In [ ]:
hashes = at.check_machine_b()
for name, digest in hashes.items():
    print(f"   {name:<24} {digest[:16]}…")
machine = at.load_machine_b()
print(f"\n{machine.parameters:,} parameters, config {machine.config}, vocabulary "
      f"{len(machine.vocabulary)} characters")
print(f"{len(machine.model.state_dict())} tensors in the state dictionary\n")
print(f"{'part':<34}{'parameters':>12}{'share':>8}")
for row in at.parameter_table(machine.model):
    print(f"{row['part']:<34}{row['parameters']:>12,}{100 * row['share']:>7.1f}%")

agree = at.corpus_agrees_with_training_report()
print(f"\ntraining corpus here: {agree['characters']:,} characters, sha256 {agree['sha256'][:16]}… — "
      f"{'the text Machine B trained on' if agree['agrees'] else 'NOT the text Machine B trained on'}")
report = machine.report
print(f"Machine B's training ran {report['steps']:,} steps in {report['elapsed_seconds'] / 60:.0f} minutes and reached "
      f"a held-out loss of {report['final_validation_loss']:.3f}")

ok = (machine.parameters == GATE["machine_b_parameters"]
      and machine.config["context"] == GATE["machine_b_context"]
      and len(machine.vocabulary) == GATE["machine_b_vocabulary"]
      and machine.config["n_layers"] == GATE["machine_b_layers"]
      and machine.config["n_heads"] == GATE["machine_b_heads"]
      and len(machine.model.state_dict()) == GATE["machine_b_state_dict_keys"]
      and len(hashes) == GATE["machine_b_files"] and agree["agrees"])
gate("C2", ok, f"{len(hashes)} files match the freeze manifest; {machine.parameters:,} parameters, "
               f"{len(machine.model.state_dict())} state-dict tensors, corpus hash agrees")

## C2b. Inside one layer: the residual, layer normalisation, the feed-forward block

9.A2's three boxes around attention, first on the slide's numbers, then inside Machine B's first layer.
**Layer normalisation** standardises one token's vector over its own $d$ entries, $(\boldsymbol x-\mu)/\sigma$,
then rescales it with learned $\boldsymbol\gamma,\boldsymbol\beta$. The **residual** adds what a block computed
to what came in. The **feed-forward block**,
$\mathbf W_2\,\mathrm{GELU}(\mathbf W_1\boldsymbol h+\boldsymbol b_1)+\boldsymbol b_2$ with $4d$ hidden units,
transforms each token on its own. Machine B normalises *before* each block, as most current models do:

$$\boldsymbol h=\boldsymbol e+\mathrm{Attn}\bigl(\mathrm{LN}_1(\boldsymbol e)\bigr),\qquad
\text{out}=\boldsymbol h+\mathrm{FFN}\bigl(\mathrm{LN}_2(\boldsymbol h)\bigr).$$

The cell rebuilds the layer from these parts and checks it against the model's own. Then it changes one
character's vector and looks at which rows move: only attention carries a change to other positions and, the
model being causal, only to later ones.

In [ ]:
ln = cx.layer_norm(cx.LN_X)
print(f"9.A2's token, x = (2, 4, 6, 8): mu = {ln['mu']:.0f}, sigma = {ln['sigma']:.2f}, (x - mu)/sigma = "
      f"({', '.join(f'{v:.2f}' for v in ln['out'])})")
inside = cx.layer_by_hand(machine, at.DEMO_SENTENCE)
print(f"\nMachine B, layer 1, on the {inside['characters']} characters of C3's sentence: d = {inside['d']}, "
      f"{inside['hidden']} hidden units, {inside['activation']}")
print(f"   the layer rebuilt from its parts, against the model's own: largest gap {inside['by_hand_vs_block']:.1e}")
print(f"   LN_1 as gamma * (e - mu)/sigma + beta, against the model's LN_1: largest gap "
      f"{inside['norm_matches_formula']:.1e}")
print(f"   the standardised part has mean 0 and standard deviation 1 (within "
      f"{max(inside['z_mean'], inside['z_std']):.0e}); the learned gamma runs from {inside['gamma'][0]:.2f} to "
      f"{inside['gamma'][1]:.2f}")
print(f"   what each block adds, as a share of its input's length (median over the characters): attention "
      f"{inside['attention_update']:.2f}, feed-forward {inside['ffn_update']:.2f}")
mb_table = at.parameter_table(machine.model)
ffn = sum(r["parameters"] for r in mb_table if "feed_forward" in r["part"])
att = sum(r["parameters"] for r in mb_table if "attention" in r["part"])
mb_layers, mb_d = machine.config["n_layers"], machine.config["d_model"]
ffn_share = 100 * ffn / machine.parameters
print(f"\nC2's table, added up. Per layer the feed-forward block has {ffn // mb_layers:,} parameters "
      f"(8d² = {8 * mb_d * mb_d:,}, plus biases)\nand attention {att // mb_layers:,} (4d² = {4 * mb_d * mb_d:,}, plus "
      f"biases). Over {mb_layers} layers the feed-forward blocks hold\n{ffn:,} of the {machine.parameters:,} parameters: "
      f"{ffn_share:.1f}%.")
probe = inside["probe"]
print(f"\nchange the vector of character {probe + 1} ('{at.DEMO_SENTENCE[probe]}'), recompute, and see which rows move:")
print(f"   layer normalisation, every other character    largest change {inside['norm_other_rows']:.1e}")
print(f"   feed-forward, every other character           largest change {inside['ffn_other_rows']:.1e}")
print(f"   attention, the {probe} characters before it        largest change {inside['attention_before_probe']:.1e}")
print(f"   attention, every character after it           smallest change {inside['attention_after_probe']:.1e}")
print("Layer normalisation and the feed-forward block work on one token at a time. Attention is the one step that")
print("moves information between tokens, and the causal mask lets it move only forward.")

printed = (f"{ln['mu']:.0f}", f"{ln['sigma']:.2f}", tuple(f"{v:.2f}" for v in ln["out"]))
gate("C2b", printed == (cx.SLIDE_LN["mu"], cx.SLIDE_LN["sigma"], cx.SLIDE_LN["out"])
     and f"{ffn_share:.1f}" == cx.SLIDE_FFN_SHARE
     and inside["by_hand_vs_block"] < 1e-5 and inside["norm_other_rows"] <= 1e-7 and inside["ffn_other_rows"] <= 1e-7
     and inside["attention_before_probe"] <= 1e-7 and inside["attention_after_probe"] > 0.0,
     "9.A2's layer norm comes back at its two decimals, and its feed-forward share at one; Machine B's layer "
     "rebuilt by hand equals its own; a "
     "changed character moves no other row except through attention, and no row before it")

## C3. What the heads look at

The rows of an attention map are query positions, the columns the positions they read. Above the diagonal the
weights are **exactly zero**: the model is causal and no position may read its own future. Each row's Shannon
entropy, in nats, says what the head is doing — $0$ is a head reading one position. A head averaging
over everything it can see scores $\ln t$ at row $t$, which reads $t$ positions, so $\ln T$ only on the last row.

In [ ]:
amap = at.attention_map(machine, at.DEMO_SENTENCE)
print(f'"{at.DEMO_SENTENCE}"  —  {amap["characters"]} characters, '
      f'{amap["n_layers"]} layers x {amap["n_heads"]} heads')
flat = float(np.mean(np.log(np.arange(2, amap["characters"] + 1))))
print(f"flat on the last row is ln {amap['characters']} = {amap['max_possible_entropy']:.2f} nats; row t reads t characters, so over")
print(f"rows 2 to {amap['characters']}, the rows each mean below averages, flat attention would score {flat:.2f}\n")
print(f"{'mean row entropy':<20}" + "".join(f"{'head ' + str(h + 1):>10}" for h in range(amap["n_heads"])))
for l, row in enumerate(amap["entropy"], start=1):
    print(f"{'layer ' + str(l):<20}" + "".join(f"{v:>10.2f}" for v in row))
print(f"\nlargest weight above the diagonal: {amap['largest_weight_above_the_diagonal']:.1e}")
print("Layer 1 stays diffuse across all four heads; by layer 2 at least one head has collapsed onto a few")
print("positions. Specialisation is learned, and it is learned in that order.")

low_l = int(np.argmin([min(r) for r in amap["entropy"]]))
low_h = int(np.argmin(amap["entropy"][low_l]))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for a, (l, h, title) in zip(ax, [(0, 0, "layer 1, head 1"),
                                 (low_l, low_h, f"layer {low_l + 1}, head {low_h + 1} (lowest entropy)")]):
    im = a.imshow(amap["maps"][l][h], cmap="magma", vmin=0, vmax=1)
    a.set(title=title, xlabel="position read", ylabel="query position")
    fig.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

rows = np.concatenate([m.reshape(-1, m.shape[-1]) for m in amap["maps"]])
row_error = float(np.abs(rows.sum(axis=1) - 1.0).max())
ok = (amap["largest_weight_above_the_diagonal"] <= GATE["attention_weight_above_diagonal_max"]
      and row_error < GATE["attention_row_sum_error_max"]
      and amap["n_layers"] == GATE["machine_b_layers"] and amap["n_heads"] == GATE["machine_b_heads"])
gate("C3", ok, f"every weight above the diagonal is {amap['largest_weight_above_the_diagonal']:.0e}; "
               f"every row sums to 1 within {row_error:.1e}")

## C4. What it writes, and the one dial

Generation is: predict the next character, divide the logits by the **temperature**, draw, append, repeat.
Below one, the distribution sharpens and the model repeats the house phrasing. At one it is the distribution
the model actually learned. Above one the tail gets its share and the spelling goes.

In [ ]:
for temperature in at.TEMPERATURES:
    text = at.generate(machine, prompt="The Committee decided", n_new=200, temperature=temperature)
    print(f"temperature {temperature}")
    print("   " + text.replace("\n", " / "))
    print()
print("One character at a time from a 0.6M-parameter model. The spelling, the house phrasing and the shape of")
print(f"a policy sentence are all learned from {machine.report['corpus_characters']:,} characters of FOMC "
      f"prose;")
print("the arithmetic of a rate decision is not, and nothing in the training objective asked for it.")

## C4b. One row, one distribution, one loop

C4 turned the dial; here is one step of what it turns, as 9.A3's "How a Decoder Writes" draws it. Give Machine B
the prefix *"The Committee decided to "*. The last row of its last layer, through the output layer, gives one
score per character of its 78-character vocabulary, and the softmax turns the scores into a distribution over the
next character. A word is a run of characters, so its probability is a product of such probabilities, one per
character. **Greedy** decoding takes the top character, appends it and asks again. **Temperature** $\tau$ divides
the scores before the softmax; **top-$p$** draws only from the smallest set of characters holding probability
$p$. The same step is in the recorded run; the cell compares the two.

In [ ]:
rec_nc = record["nextchar"]
prefix = rec_nc["prefix"]
logits = cx.next_char_logits(machine, prefix)
probs = at.softmax(logits)
top = np.argsort(probs)[::-1][:10]
print(f"after {prefix!r}, the ten likeliest next characters:")
print("   " + "   ".join(f"{machine.vocabulary[i]!r} {probs[i]:.3f}" for i in top))
print("\nthe next word, one character at a time (the word, then a space):")
for word in cx.DECODER_WORDS:
    p_word = cx.word_probability(machine, prefix, word)
    print(f"   {word:<10}" + (f"{p_word:.4f}" if p_word >= 1e-3 else f"{p_word:.1e}"))
print(f"9.A3 draws this distribution as a schematic; here it is measured, for a model of {machine.parameters:,} "
      "parameters.\n")
for tau in (0.5, 1.0, 2.0):
    pt = cx.at_temperature(logits, tau)
    print(f"tau = {tau}: P('m') = {pt[machine.stoi['m']]:.3f}, entropy {cx.entropy(pt):.2f} nats; top-p with "
          f"p = 0.9 keeps {cx.top_p_size(pt)} of {len(pt)} characters")
text = cx.greedy(machine, prefix, n_new=60)
print(f"\ngreedy, sixty characters: {text!r}")
for tau, recorded_text in rec_nc["sampled"].items():
    live = at.generate(machine, prefix, n_new=60, temperature=float(tau))
    print(f"sampled at tau = {tau}, seed {at.SEED}: {live!r}"
          + ("" if live == recorded_text else f"\n   the recorded draw: {recorded_text!r}"))
print("Greedy repeats the house phrasing. A high temperature spends probability on characters the model doubts.")

gap = float(np.abs(np.round(logits, 5) - np.array(rec_nc["logits"])).max())
gate("C4b", gap < 1e-4 and [machine.vocabulary[i] for i in top] == [r["char"] for r in rec_nc["top"]]
     and text == rec_nc["greedy"],
     f"the {len(logits)} scores match the recorded run (largest gap {gap:.0e}); the same top ten; the same "
     "greedy text")

## C5. Sixty steps from noise

The checkpoint above is a result. This is the process. A **freshly initialised** model of the same
architecture is trained on the same corpus for sixty steps with Machine B's own recipe — AdamW, lr $3\times10^{-4}$,
linear warm-up over 100 steps, gradients clipped at 1.0, batches of 32 windows of 128 characters.

Sixty steps is not a small version of Machine B's three thousand; it is the first inch of them. The point is
the starting value: a model that knows nothing picks uniformly among 78 characters and pays
$\ln 78 = 4.357$ nats per character. Watch it leave that line.

The held-out split is by **whole statement** — every twentieth one — never by random window, so a validation
loss cannot be read off a sentence the model already met in training.

In [ ]:
t0 = time.time()
fresh = at.train_a_few_steps(machine, n_steps=60)
print(f"{fresh['n_steps']} steps in {time.time() - t0:.0f} s on CPU, {fresh['parameters']:,} parameters "
      f"from a fresh initialisation")
print(f"loss: uniform {fresh['uniform_loss']:.3f}  ->  step 1 {fresh['first_loss']:.3f}  ->  "
      f"step {fresh['n_steps']} {fresh['last_loss']:.3f}   (held out: "
      f"{fresh['first_validation']:.3f} -> {fresh['last_validation']:.3f})")
d = fresh["data"]
print(f"split: {d['n_held_out']} of {d['n_statements']} statements held out, "
      f"{d['train_characters']:,} training characters and {d['validation_characters']:,} held-out")

steps, losses = zip(*fresh["train_curve"])
vsteps, vlosses = zip(*fresh["validation_curve"])
plt.figure(figsize=(6, 3.2))
plt.plot(steps, losses, color=MUTED, lw=0.9, label="training batch")
plt.plot(vsteps, vlosses, "o-", color=TEAL, ms=3, label="held-out statements")
plt.axhline(fresh["uniform_loss"], color=RED, ls="--", lw=0.9,
            label=f"uniform over 78 characters, ln 78 = {fresh['uniform_loss']:.2f}")
plt.axhline(machine.report["final_validation_loss"], color="0.6", ls=":", lw=0.9,
            label=f"Machine B after {machine.report['steps']:,} steps")
plt.xlabel("step"); plt.ylabel("cross-entropy, nats per character"); plt.legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()
print("after 60 steps, the fresh model writes:")
print("   " + fresh["sample_after"].replace("\n", " / "))
print("It has learned the letter frequencies and the space. It has not learned a word.")

# Every character of every statement has to land in exactly one of the two streams. Each stream is
# the statements joined by a blank line, so subtract the separators before comparing the totals.
n_train = d["n_statements"] - d["n_held_out"]
bodies = d["characters"] - 2 * (d["n_statements"] - 1)
split_bodies = ((d["train_characters"] - 2 * (n_train - 1))
                + (d["validation_characters"] - 2 * (d["n_held_out"] - 1)))
accounting = bodies - split_bodies
print(f"characters in the {d['n_statements']} statement bodies: {bodies:,}; in the two streams: "
      f"{split_bodies:,}; unaccounted for: {accounting}")
ok = (fresh["parameters"] == GATE["fresh_model_parameters"]
      and d["n_held_out"] == GATE["fresh_model_held_out_statements"]
      and abs(accounting) <= GATE["fresh_model_character_accounting_error_max"])
gate("C4", ok, f"the fresh model is the same architecture ({fresh['parameters']:,} parameters); "
               f"{d['n_held_out']} whole statements held out and every character accounted for")

## C6. An encoder reads: fill the blank, pool a passage, label a sentence

9.A3's encoder, BERT, is pretrained to fill in hidden words. In research it reads text and returns one number
per sentence or passage: a label from a fitted head, or a cosine between pooled vectors ("Encoders in Economic
Research"). Three recorded runs show the three uses and what each number means. **Fill the blank**:
multilingual BERT's own pretraining head guesses a masked word. **Pool**: average a passage's last-layer
vectors, or take its [CLS] vector, and compare passages by cosine. **Label**: FinBERT, BERT with a three-way
head fine-tuned on the Financial PhraseBank (sentences of financial news labelled positive, negative or
neutral), labels five sentences in the Committee's style. As in C0, the numbers are read from the record; no
model is called.

In [ ]:
fm = record["fillmask"]
print(f"fill the blank: {fm['model']}, {fm['head']}")
for i, s in enumerate(fm["sentences"]):
    print(f"   {i + 1}. {s['text']}")
    print("      top guesses: " + ", ".join(f"{t['token']} {t['p']:.3f}" for t in s["top"][:6]))
print("the watch list, the probability of each word in the blank:")
print(pd.DataFrame({f"sentence {i + 1}": s["watch"] for i, s in enumerate(fm["sentences"])})
      .to_string(float_format=lambda x: f"{x:.4f}"))
infl, unemp = fm["sentences"][1]["watch"], fm["sentences"][2]["watch"]
print(f"After rising inflation and after rising unemployment (sentences 2 and 3): 'increase' {infl['increase']:.4f} "
      f"and {unemp['increase']:.4f},")
print(f"'raise' {infl['raise']:.4f} and {unemp['raise']:.4f}. The guesses are about English, not about policy.")

pool = record["pooling"]
print(f"\npool: {pool['model']}, the {pool['layer']} layer")
for k, para in pool["paragraphs"].items():
    print(f"   {k}  {textwrap.shorten(para['text'], 104)}")
print(f"   {'pair':<6}{'mean of the token vectors':>27}{'[CLS] vector':>15}")
for pair, c in pool["cosines"].items():
    print(f"   {pair:<6}{c['mean']:>27.3f}{c['cls']:>15.3f}")
print(f"Mean pooling sets the museum paragraph apart. The raw [CLS] vector scores every pair above "
      f"{min(c['cls'] for c in pool['cosines'].values()):.2f}: before a head")
print("is fitted on a task, [CLS] is no summary. And a cosine measures nearness, not sameness: a central bank's")
print(f"decision and a company's financing costs score {pool['cosines']['AB']['mean']:.2f}.")

fb = record["finbert"]
print(f"\nlabel: {fb['model']}: positive, negative or neutral financial news")
for s in fb["sentences"]:
    print(f"   {s['label']:<9}" + "  ".join(f"{k} {v:.2f}" for k, v in s["probs"].items()) + f"   {s['text']}")
print("A rate increase reads as neutral news and a cut as negative: FinBERT measures the sentiment its training")
print("labels carry, not policy stance. What a number means is set by the training labels and the corpus.")

# Block D — The recorded language-model instrument

A language model asked to label a statement is an **instrument**. Like any instrument it has a reading, and
like any instrument the reading is worthless without provenance and without a measure of how much it moves
when nothing it is measuring has moved.

Four prompt variants were recorded outside this notebook, by a separate instructor process, on 2026-09-20:

| variant | what it is | statements |
|---|---|---|
| **A** | the frozen prompt | all 245 |
| **B** | the same task, **reworded** | 48 |
| **C** | the same words, **options reordered** (DOVISH first) | 48 |
| **D** | **byte-identical to A**, run again | 48 |

A against B and C measures prompt sensitivity. A against D measures nothing but **sampling noise**: the input
was the same bytes. This notebook reads the record; it never calls a model and never touches the network.

In [ ]:
STANCE = ASSETS / "llm_stance"
lines = (STANCE / "raw_replies.jsonl").read_text(encoding="utf-8").splitlines()
replies = pd.DataFrame([json.loads(line) for line in lines if line.strip()])
replies["model"] = replies["model_used"].map(lambda v: v[0] if isinstance(v, list) else v)

print("provenance of every score in this block")
print(f"   model returned   : {sorted(replies['model'].unique())}")
print(f"   harness          : {sorted(replies['harness'].unique())}")
print(f"   recorded         : {replies['recorded_at'].min()[:10]}, "
      f"{replies['recorded_at'].min()[11:]} to {replies['recorded_at'].max()[11:]}")
print(f"   replies          : {len(replies)}   unparsed or errored: {int(replies['error'].notna().sum())}"
      f"   cost: ${replies['cost_usd'].sum():.2f}")
print("   prompt sha256-16 : " +
      "   ".join(f"{v} {h}" for v, h in replies.groupby("variant")["prompt_sha256_16"].first().items()))
print("                      A and D carry one hash because D is A's bytes, run a second time.\n")

print("the first eight variant-A scores, each beside what produced it:")
print(f"{'date':<12}{'label':<10}{'model':<18}{'harness':<24}{'recorded':<12}prompt")
for _, r in replies[replies["variant"] == "A"].sort_values("date").head(8).iterrows():
    print(f"{r['date']:<12}{r['label']:<10}{r['model']:<18}{r['harness']:<24}"
          f"{r['recorded_at'][:10]:<12}{r['prompt_sha256_16']}")
print("\nNo number a model produced appears anywhere in Session 9 without those four fields beside it.")

by_variant = replies.groupby("variant")["prompt_sha256_16"].first()
a, dvar = GATE["llm_identical_prompt_pair"]
ok = (len(replies) == GATE["llm_replies"]
      and replies["variant"].nunique() == GATE["llm_variants"]
      and int(replies["error"].notna().sum()) <= GATE["llm_errors_max"]
      and all(replies[f].notna().all() for f in GATE["llm_reply_fields"])
      and by_variant[a] == by_variant[dvar]
      and len({by_variant[a], by_variant["B"], by_variant["C"]}) == 3)
gate("D1", ok, f"{len(replies)} replies, none errored, each carrying "
               f"{', '.join(GATE['llm_reply_fields'])}; {a} and {dvar} share one prompt hash and "
               f"B and C each differ")

## D2. Agreement, and Cohen's κ

Raw agreement flatters an instrument that mostly says the same thing. **Cohen's κ** subtracts the agreement
two raters would reach by chance given their own label frequencies: $\kappa = (p_o - p_e)/(1 - p_e)$. The last column reads κ
on Fleiss's (1981) bands: below 0.40 poor, 0.40 to 0.75 fair to good, above 0.75 excellent. E3 adds Landis
and Koch's (1977) finer bands, which have no cut at 0.75: 0.61 to 0.80 substantial, 0.81 to 1.00 almost
perfect. Both are conventions, not tests.

κ is computed here by `cohen_kappa` from `book_code/ch14/human_llm_kappa.py` — the chapter's own function,
hash-checked against `book_code/MANIFEST.json` before it is allowed to run.

In [ ]:
scores = pd.read_csv(STANCE / "scores.csv")
wide = scores.pivot(index="date", columns="variant", values="label")
common = wide.dropna()
hk = me.book_module("human_llm_kappa")             # the chapter's own kappa, hash-checked on import

what = {"AB": "reworded prompt", "AC": "options reordered", "AD": "byte-identical repeat",
        "BC": "reworded vs reordered", "BD": "reworded vs repeat", "CD": "reordered vs repeat"}
rows = []
for p, q in itertools.combinations(sorted(wide.columns), 2):
    u, v = common[p].to_numpy(), common[q].to_numpy()
    rows.append(dict(pair=f"{p} vs {q}", what=what[p + q], agreement=float((u == v).mean()),
                     kappa=float(hk.cohen_kappa(u, v))))
pairs = pd.DataFrame(rows)
print(f"all four variants cover the same {len(common)} statements\n")
print(f"{'pair':<10}{'what it varies':<26}{'agreement':>11}{'kappa':>9}   {'Fleiss band'}")
for r in rows:
    print(f"{r['pair']:<10}{r['what']:<26}{100 * r['agreement']:>10.1f}%{r['kappa']:>9.3f}   "
          f"{hk.fleiss_band(r['kappa'])}")

ad = pairs.set_index("pair").loc["A vs D"]
ab = pairs.set_index("pair").loc["A vs B"]
print(f"\nThe byte-identical repeat, A vs D, agrees LESS than the reworded prompt, A vs B: "
      f"kappa {ad['kappa']:.3f} against {ab['kappa']:.3f}.")
print("On this corpus, with this model, on this day, resampling the same prompt moves the label at least as")
print("much as rewording it. Sampling noise is as large as wording here.")

ok = (len(common) == GATE["llm_common_dates"]
      and sorted(pd.unique(scores["label"])) == GATE["llm_labels"]
      and int((scores["variant"] == "A").sum()) == GATE["llm_variant_a_scored"]
      and len(wide.columns) == GATE["llm_variants"])
gate("D2", ok, f"the four variants are compared on the same {len(common)} statements, "
               f"labels confined to {GATE['llm_labels']}, variant A covering all "
               f"{GATE['llm_variant_a_scored']}")

In [ ]:
print("where A and D disagree (rows: A's label; columns: the same prompt, run again)")
table = pd.crosstab(common["A"], common["D"])
print(table.to_string())
moved = int(len(common) - np.trace(table.to_numpy()))
print(f"\n{moved} of {len(common)} statements changed label when nothing about the input changed.")
print("A researcher who ran this prompt once, and reported the labels, would be reporting one draw from")
print("that table. Neither column is more correct than the other; that is the point.")
print("\nRead the ordering carefully. It says sampling noise is large HERE — 48 statements, one model, one")
print("day, three coarse labels. It is not a general claim that wording does not matter, and a study with a")
print("bigger wording contrast or a finer label set could easily reverse it. What it does establish is that")
print("prompt-sensitivity checks that never repeat the same prompt cannot tell the two apart at all.")

## D3. Does the reading line up with what the Committee then did?

Same validation as block B: the score against the subsequent policy path, per cycle, inside the training
window so that it is comparable with the other signals.

In [ ]:
llm_signal = tx.signal_llm_stance(statements)          # reads scores.csv; variant A; +1 / 0 / -1
llm_check = tx.validate_signal(dates, llm_signal["score"], "language-model stance",
                               horizon=GATE["validation_horizon"])
print(f"{llm_signal['n_scored']} of {llm_signal['n_statements']} statements carry a variant-A score\n")
print(f"{'signal':<26}{'hiking mean':>13}{'cutting mean':>14}{'rank corr. w/ path':>21}{'verdict':>10}")
for v in (llm_check, checks[1], checks[0]):
    print(f"{v['signal']:<26}{v['hiking_mean']:>13.3f}{v['cutting_mean']:>14.3f}"
          f"{v['rank_correlation_with_subsequent_path']:>21.3f}"
          f"{'PASSES' if v['passes'] else 'FAILS':>10}")
print("\nper cycle, over the cycles the training window covers:")
per_cycle = tx.validation_table([llm_check]).drop(columns="signal").dropna(subset=["mean"])
print(per_cycle.to_string(index=False))

r = REC["text"]["validation"]["language-model stance | training window"]
ok = (abs(llm_check["rank_correlation_with_subsequent_path"]
          - r["rank_correlation_with_subsequent_path"]) < GATE["validation_tolerance"]
      and llm_check["passes"] == r["passes"] and llm_check["n_scored"] == r["n_scored"])
gate("D3", ok, f"the recorded instrument reproduces the recorded run to "
               f"{GATE['validation_tolerance']:g}: rank correlation "
               f"{llm_check['rank_correlation_with_subsequent_path']:+.3f} on "
               f"{llm_check['n_scored']} statements")

## D4. What has been established, and what has not

Two things were measured in this block, and neither of them is validity.

1. **Consistency.** How much the reading moves when the instrument is re-applied to the same object, and when
   the instrument is reworded. Both moved about one label in six, the repeat slightly more than the rewording.
2. **Convergent validity.** Whether the reading lines up with an independent quantity it ought to line up
   with — here, the Fed's own subsequent policy path. It does, better than the embedding axis and less well
   than the decision-sentence regex (B3).

What is **not** here is validity against a gold standard. That would require a set of statements labelled by
people, under a written coding manual, with their disagreements recorded — and **no human gold labels exist in
this build**. Nobody hand-coded these 245 statements. Every κ in this block is model against model.

Without gold labels there is no way to say whether the instrument is *biased*, only whether it is *stable* and
whether it *agrees with something else*. An instrument can be perfectly stable and consistently wrong; block D
cannot tell those apart. Block E is what buying the missing piece costs, and what it buys.

# Block E — What validation buys

Suppose the reading goes into a regression as a regressor: $y_i = \alpha + \beta m^{*}_i + \gamma' c_i + u_i$,
with $\hat m_i = m^{*}_i + \varepsilon_i$ in place of $m^{*}_i$. Chapter 14 works this through, and the four
scripts of that chapter are vendored byte-identically under `book_code/ch14/`. The cells below **import those
scripts and call their own functions in their own order** — nothing here re-derives a statistic, and a hash
check stops the run if a vendored file has been edited.

Each block is gated against the digits printed in `chapters/chapter14.tex`, at the precision the chapter
prints them. A disagreement would be recorded and left standing; nothing in `session05_measure.py` is tuned
towards the book.

## E1. Four estimators of the same $\beta$

In [ ]:
bias = me.bias_correction_table()
print(f"truth: beta = {bias.beta_true:.2f} on n = {bias.n:,} documents, of which "
      f"{bias.n_validation} are hand-coded")
print(f"corr(m*, m-hat) = {bias.corr_mstar_mhat:.3f}; mean error {bias.mean_bias:+.4f} — close to zero")
print(f"but the error loads on a covariate: slope on size {bias.error_on_size_slope:+.4f} "
      f"(s.e. {bias.error_on_size_se:.4f})\n")
print(f"{'estimator':<20}{'beta':>9}{'s.e.':>9}{'95% interval':>22}{'width':>9}{'% of truth':>12}")
for name, row in bias.estimators.items():
    print(f"{name:<20}{row['beta']:>9.4f}{row['se']:>9.4f}"
          f"{'[' + format(row['ci_low'], '.3f') + ', ' + format(row['ci_high'], '.3f') + ']':>22}"
          f"{row['ci_width']:>9.4f}{row['pct_of_truth']:>11.0f}%")
c = bias.calibration
print(f"\ncalibration on the {bias.n_validation} hand-coded rows: "
      f"m* = {c['intercept']:.3f} + {c['slope_m_hat']:.3f} m-hat + {c['slope_c']:.3f} c")
print(f"the corrected interval is {bias.corrected_narrower_pct:.0f}% narrower than using the hand-coded "
      f"rows alone.")
print("\nRead the table downward. Naive is what a researcher gets by treating the model's score as the")
print("variable: attenuated, and the mean error being near zero did not save it. Validation-only is")
print("unbiased and imprecise. Corrected calibrates the score on the hand-coded rows and then uses it on")
print("the full sample, buying the precision back.")

fig, ax = plt.subplots(figsize=(6.4, 2.6))
for i, (name, row) in enumerate(bias.estimators.items()):
    ax.plot([row["ci_low"], row["ci_high"]], [i, i], color=MUTED, lw=2)
    ax.plot(row["beta"], i, "o", color=TEAL, ms=5)
ax.axvline(bias.beta_true, color=RED, ls="--", lw=1, label=f"truth, {bias.beta_true:.2f}")
ax.set_yticks(range(len(bias.estimators))); ax.set_yticklabels(list(bias.estimators))
ax.invert_yaxis(); ax.set_xlabel("beta"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## E2. How many hand-coded documents is enough

The width of the corrected interval against the size of the hand-coded set. The curve falls steeply, then
slowly: past a few hundred documents the binding constraint stops being the calibration equation's own
error and becomes the full-sample residual variance. That is the budget rule.

In [ ]:
sweep = me.validation_size_sweep()
print(f"{'hand-coded n':>14}{'beta':>10}{'95% interval':>22}{'width':>9}{'calibration slope':>20}")
for row in sweep.rows:
    print(f"{row['n_validation']:>14}{row['beta_corrected']:>10.4f}"
          f"{'[' + format(row['ci_low'], '.3f') + ', ' + format(row['ci_high'], '.3f') + ']':>22}"
          f"{row['ci_width']:>9.4f}{row['calibration_slope_m_hat']:>20.4f}")
print(f"\n50 -> 200 labels narrows the interval by {sweep.narrowing_50_to_200_pct:.0f}%; "
      f"200 -> 800 by a further {sweep.narrowing_200_to_800_pct:.0f}%.")
print("Four times the hand-coding for a fifth of the gain: the second four hundred documents are the")
print("expensive ones, and a design calculation is how you find that out before paying for them.")

plt.figure(figsize=(5.4, 2.8))
plt.plot([r["n_validation"] for r in sweep.rows], [r["ci_width"] for r in sweep.rows],
         "o-", color=TEAL, ms=4)
plt.xscale("log"); plt.xlabel("hand-coded documents"); plt.ylabel("width of the 95% interval")
plt.tight_layout(); plt.show()

digits = bias.gates + sweep.gates
gate("E1", all(g.agrees for g in digits) and len(digits) == GATE["chapter14_digits_estimation"],
     f"{sum(g.agrees for g in digits)} of {len(digits)} digits printed in chapter 14 reproduce here")

## E3. An overall κ that hides the failure

Chapter 14's table 14.2: a model and a human coder on four narrative categories. Overall κ reads as excellent
on the Fleiss bands. One class — the rare one, which is where the interesting hypothesis usually lives — is
coded far worse, and the overall figure cannot show it. Merging that class upward raises the overall number
without improving a single coding decision: it removes the distinction the research question was about.

In [ ]:
pck = me.per_class_kappa()
print(f"{pck.n_audit} hand-coded rows out of {pck.n_full:,}; accuracy {100 * pck.accuracy:.0f}%, "
      f"overall kappa {pck.kappa_overall:.3f}")
print(f"which Fleiss calls '{pck.fleiss_band}' and Landis-Koch '{pck.landis_koch_band}'\n")
print(f"{'class':<18}{'share':>8}{'support':>9}{'recall':>9}{'kappa':>9}   above 0.75?")
for row in pck.classes:
    print(f"{row['label']:<18}{100 * row['share']:>7.1f}%{row['support']:>9}{row['recall']:>9.2f}"
          f"{row['kappa']:>9.3f}   {'yes' if row['meets_threshold_0_75'] else 'NO'}")
m = pck.merge
print(f"\nmerging '{m['from']}' into '{m['into']}': kappa {m['kappa_before']:.3f} -> "
      f"{m['kappa_after']:.3f}")
print(f"what it costs: {m['what_it_costs']}")

## E4. A screen that cannot see the errors you most fear

Self-consistency screening asks the model the same question several times and flags the claims whose answers
move: a claim's inconsistency $h$ is one minus the mean overlap (Jaccard) between its first answer and each
resample, $0$ for a claim answered the same way every time, and the screen flags a claim whose $h$ passes a
threshold. Read the last line of the output carefully: resampling measures **how much the answer moves**, not
whether it is true. A false statement the model holds firmly resamples as consistently as a true one.

In [ ]:
sc = me.self_consistency()
print(f"{sc.n_claims} claims, {sc.resamples} resamples each\n")
print(f"{'kind of claim':<22}{'n':>6}{'mean h':>10}{'s.d.':>8}{'p10':>8}{'p90':>8}")
for name, row in sc.by_kind.items():
    print(f"{name:<22}{row['n']:>6}{row['mean_h']:>10.3f}{row['sd_h']:>8.3f}{row['p10']:>8.3f}"
          f"{row['p90']:>8.3f}")
print(f"\nbest threshold {sc.threshold:.2f}: precision {sc.precision:.2f}, recall {sc.recall:.2f}, "
      f"F1 {sc.f1:.2f}, flagging {100 * sc.flagged_share:.0f}% of claims")
print(f"fabricated claims missed: {sc.fabricated_missed} of {sc.fabricated_total}")
print(f"CONFIDENTLY WRONG claims — held firmly, and false — that the screen never flags: "
      f"{sc.stable_error_undetected} of {sc.stable_error_total}, {sc.stable_error_undetected_pct:.1f}%")
print("\nRaising the number of resamples does not help. It sharpens the score; it does not change what the")
print("score measures. The screen is a triage tool, not a test of truth.")

## E5. Design: how many items before an order effect can be told from zero

Two questions, one calculation. Which presentation removes a first-position effect — and how many items are
needed before the effect can be distinguished from zero.

The effect is how much more often the model picks the option listed first than the 0.50 it would if position
did not matter. Its size depends on how the options are labelled: the chapter takes 0.035 with letter labels,
0.234 with symbol labels and 0.132 overall from Brucks and Toubia (2025).

In [ ]:
oe = me.order_effect_power()
print(f"{'design':<16}{'mean bias':>12}{'RMSE':>10}")
for name, row in oe.designs.items():
    print(f"{name:<16}{row['mean_bias']:>12.4f}{row['rmse']:>10.4f}")
print("\nHolding the order fixed leaves the whole effect standing. Randomising the order across items removes")
print("it. Exhausting the orderings at a fixed budget adds nothing: as unbiased, and no more precise.\n")
print(f"{'effect size':>12}   {'setting':<17}{'items for 80% power':>22}{'simulated power':>18}")
for row in oe.deltas:
    print(f"{row['delta']:>12.3f}   {(row['setting'] or '—'):<17}{row['n_required_80pct_power']:>22,}"
          f"{row['simulated_power_at_n']:>18.2f}")
print(f"\npower at 50 items against the letter-label effect: {oe.letter_power_at_50_items:.2f}")
print("The requirement swings by a factor of more than forty across two label formats. A robustness check run on a few")
print("dozen items is evidence of nothing, in either direction.")

digits = pck.gates + sc.gates + oe.gates
gate("E2", all(g.agrees for g in digits) and len(digits) == GATE["chapter14_digits_agreement"],
     f"{sum(g.agrees for g in digits)} of {len(digits)} digits printed in chapter 14 reproduce here "
     f"({GATE['chapter14_digits_total']} across the block)")

# Block F — Does the Committee's language improve the forecast?

Session 7 built a one-month-ahead forecast of the 10-year Treasury yield from numbers alone. Block F asks
whether the FOMC's own words, available at the moment the forecast is made, improve on it.

**The design was written out in full and hashed before the test window was scored once.**
`protocol_forecast.md` is that design; `protocol_forecast.sha256` carries the hash and the moment it was
taken. The hash exists so that a reader can tell the difference between a design and a rationalisation.
The protocol says, in section 10.2, that "text adds nothing" is an acceptable, expected and fully reportable
outcome, and that **no gate anywhere in block F depends on text helping**.

## F1. The inputs, and the bar to beat

In [ ]:
machine_b = fc.assert_no_machine_b()          # protocol section 8.2
protocol_hash = fc.file_sha256(fc.PROTOCOL)
taken = [ln for ln in fc.PROTOCOL_SHA.read_text(encoding="utf-8").splitlines() if ln.startswith("Taken")]
print(f"protocol_forecast.md  sha256 {protocol_hash[:32]}…")
print(f"   matches the hash recorded beside it: {protocol_hash == GATE['forecast_protocol_sha256']}")
print(f"   {taken[0] if taken else ''}")
print(f"   that hash predates the first scored run: "
      f"{REC['forecast']['_meta']['protocol_hash_predates_first_scored_run']}\n")

inputs = fc.load_inputs()                     # both files hash-asserted at load; read-only
for name, digest in inputs.hashes.items():
    print(f"   {name:<26} {digest[:32]}…  matches protocol: "
          f"{digest == GATE['forecast_input_sha256'][name]}")
print(f"\npanel {len(inputs.panel)} rows, frozen forecasts {len(inputs.baseline)} rows")
print(f"the panel has no row for {', '.join(inputs.missing_forecast_dates)} — the only month-end missing "
      f"anywhere in it.")
print("Block F inherits that gap and never fills it. An interpolated month would be a month Session 7's")
print("published baselines were never scored on, and the two would stop being comparable.\n")

frozen = fc.frozen_baseline_table(inputs.baseline)
print(f"{'Session 7, scored as shipped':<30}{'RMSE':>9}{'MAE':>9}   reproduces to four decimals")
for name in ("random_walk", "ar1", "ols"):
    s = frozen["scored"][name]
    print(f"{name:<30}{s['rmse']:>9.4f}{s['mae']:>9.4f}   "
          f"{'yes' if frozen['reproduces_to_four_decimals'][name] else 'NO'}")
mlp = frozen["mlp"]
print(f"{'MLP, as Session 7 publishes it':<30}{mlp['session02_published']['rmse']:>9.4f}"
      f"{mlp['session02_published']['mae']:>9.4f}   mean of ten per-seed RMSEs")
print(f"{'MLP, the averaged column':<30}{mlp['column_as_scored']['rmse_here']:>9.4f}"
      f"{mlp['column_as_scored']['mae_here']:>9.4f}   mean forecast across ten seeds")
print("\nTwo different objects, gated against neither: averaging ten forecasts beats the average of the ten")
print("models. This lab quotes Session 7's published definition as the bar. Note what that bar is — the")
print("random walk and the AR(1) beat OLS-5 on this window, so any text model has to improve on a very")
print("low-variance forecast, not on the weakest structured model available.")

ok = (all(inputs.hashes[n] == GATE["forecast_input_sha256"][n] for n in inputs.hashes)
      and protocol_hash == GATE["forecast_protocol_sha256"]
      and inputs.missing_forecast_dates == GATE["forecast_missing_forecast_dates"]
      and all(frozen["reproduces_to_four_decimals"].values()))
gate("F1", ok, "both Session 7 files match the protocol's hashes, the 2025-11-30 gap is present and "
               "unfilled, and the three reproducible baselines come back to four decimals")

## F2. The estimation sample, the information rule, and the fitting discipline

The panel starts in 1989; the first statement is dated 1994-02-04. Fifty of Session 7's 300 training rows
predate every statement and could carry no text signal under any rule. The protocol resolves that by
**restriction, not imputation**: the estimation sample is every row with `forecast_made >= 1994-02-28`, which
leaves 250 / 60 / 79. Because the comparator is no longer estimated on what Session 7 estimated it on, every
result is reported in **three columns** — frozen file, re-estimated, plus text — so that the cost of the
restriction is never read as a text effect.

**The information rule.** Month $t$'s signal is the last statement dated on or before $t$'s last observation
date. Seven months carry two or three statements and the last one wins. Five releases are hand-listed and
removed as not being policy statements at all. In a month with no meeting the stance is carried forward,
novelty is **zero** — there is no new statement to be novel — and months-since-statement enters as its own
regressor so that staleness is measured rather than smuggled.

In [ ]:
corpus = fc.load_statements()                 # hash-asserted file by file; exclusion set applied
print(f"{len(corpus.dates)} of {GATE['corpus_statements']} releases kept; "
      f"{len(corpus.excluded)} hand-listed and removed:")
for e in corpus.excluded:
    print(f"   {e['date']}  " + textwrap.fill(e["reason"], width=94, subsequent_indent=" " * 15))

frame = fc.merge_text(inputs.panel, corpus)                # information rule; asserts no look-ahead
space = fc.fit_text_space(corpus)                          # vocabulary, IDF, SVD, seed axis
frame = fc.attach_features(frame, corpus, space)
parts = fc.split(frame)
sizes = {k: len(v) for k, v in parts.items()}
print(f"\nestimation sample from {fc.SAMPLE_START}, the first month-end after the first statement: the "
      f"panel's {int((inputs.panel['date'] < fc.SAMPLE_START).sum())} earlier rows have no statement to "
      f"carry and are dropped, not filled")
print(f"splits {sizes}")
print(f"text space: {space.vocabulary_size:,} terms from {space.n_training_documents} documents dated on "
      f"or before {space.training_cutoff}")
print(f"seed axis: {len(space.seeds_found['hawkish'])} hawkish and {len(space.seeds_found['dovish'])} "
      f"dovish terms found in that vocabulary")
print(f"stale rows: train {int(parts['train']['is_stale'].sum())}, valid "
      f"{int(parts['valid']['is_stale'].sum())}, test {int(parts['test']['is_stale'].sum())} of 79")
print(f"novelty in a stale month: {float(frame.loc[frame['is_stale'], 'novelty'].abs().max()):.1f}\n")

latest_fitted = max(d for d in corpus.dates if d <= pd.Timestamp(fc.TEXT_TRAIN_END))
print(f"latest document inside any fitted object: {latest_fitted.date()}  "
      f"(cutoff {fc.TEXT_TRAIN_END})")
print(f"latest statement reaching any row: {frame['statement_date'].max().date()}, and no row carries a "
      f"statement dated after its own forecast date: "
      f"{bool((frame['statement_date'] <= frame['date']).all())}")
print(f"Machine B: artefacts loaded = {machine_b['machine_b_artefacts_loaded']}, "
      f"Machine B modules in this process = {machine_b['session01_modules_in_process']}")
print("Block C opened Machine B in this kernel, and that object is not among block F's declared inputs:")
print("   declared inputs: baseline_predictions.csv, yield_panel.csv, fomc_manifest.json, scores.csv")
print(f"   excluded by assertion: {', '.join(fc.MACHINE_B_ARTEFACTS)}")
print("Machine B was trained on the 245 statements less every twentieth, the test window's among them: anything")
print("derived from it would be contaminated end to end, invisibly to any forecast metric.")

ok = (sizes == GATE["forecast_split_sizes"]
      and len(corpus.excluded) == GATE["forecast_excluded_releases"]
      and space.training_cutoff == GATE["forecast_text_training_cutoff"]
      and latest_fitted <= pd.Timestamp(GATE["forecast_text_training_cutoff"])
      and bool((frame["statement_date"] <= frame["date"]).all())
      and float(frame.loc[frame["is_stale"], "novelty"].abs().max()) == 0.0
      and not machine_b["machine_b_artefacts_loaded"])
gate("F2", ok, f"splits {sizes}; nothing after {fc.TEXT_TRAIN_END} in a fitted object; no statement "
               f"dated after its own row; novelty zero in every stale month; no Machine B artefact")

## F3. Scored once

The validation metrics are recomputed live — they are cheap and carry no inferential weight. The test window
is scored in **one pass**, after every choice has been made, and `score_test_once` refuses a second call in
the same process. The numbers it returns are then asserted equal to `recorded_forecast.json`: a mismatch would
mean the window had been scored again under different choices, which is the one thing the protocol forbids.

In [ ]:
runs = fc.run_models(parts, corpus, space)     # the only choice made here is the ridge penalty, on 2015-2019
valid = fc.validation_metrics(runs, parts)
print(f"{'model':<44}{'validation RMSE':>17}{'validation MAE':>16}")
for key, m_ in valid.items():
    print(f"{runs[key].name:<44}{m_['rmse']:>17.4f}{m_['mae']:>16.4f}")
for key in ("tfidf_ridge_alone", "ols5_plus_tfidf_ridge"):
    d_ = runs[key].detail
    print(f"   {runs[key].name}: alpha {d_['alpha_chosen_on_validation']:g} chosen on 2015-2019 from "
          f"{sorted(float(a) for a in d_['validation_rmse_by_alpha'])}, {d_['n_columns']} columns")

test_scores = fc.score_test_once(runs, parts)               # one pass, after every choice
scored_once = fc._TEST_SCORED["count"] == GATE["forecast_test_scored_times"]
verdict = fc.assert_matches_record(test_scores, places=GATE["forecast_record_decimals"])
try:
    fc.score_test_once(runs, parts)
    refused = False
except RuntimeError as exc:
    refused = True
    print(f"\na second scoring attempt is refused: {exc}")
print(f"test-window numbers equal recorded_forecast.json to {GATE['forecast_record_decimals']} places: "
      f"{all(verdict.values())} ({len(verdict)} models)")

gate("F3", scored_once and refused and all(verdict.values()),
     f"the test window was scored once, a second attempt raised, and all {len(verdict)} models match "
     f"the recorded file")

## F4. The answer, in three columns

`frozen file` is Session 7's published forecast scored as shipped. `re-estimated` is the same model form
refitted on the 1994-onward rows. `plus text` adds the text regressors. The gap between the first two columns
is the cost of the sample restriction and has nothing to do with text; reporting it in its own column is what
stops the sum of two changes being attributed to one of them.

In [ ]:
print(f"{'model':<24}{'frozen file':>20}{'re-estimated':>20}{'plus text':>20}")
print(f"{'':<24}{'RMSE    MAE':>20}{'RMSE    MAE':>20}{'RMSE    MAE':>20}")
for row in fc.three_column_table(frozen, test_scores):
    cells = []
    for key in ("frozen_file", "re_estimated", "plus_text"):
        cell = row[key]
        cells.append(f"{'--':>8}{'--':>8}" if cell is None
                     else f"{cell['rmse']:>8.4f}{cell['mae']:>8.4f}")
    print(f"{row['model']:<24}" + "".join(f"{c:>20}" for c in cells))

actual = parts["test"]["y10_next"].to_numpy()
err = lambda k: actual - runs[k].predictions["test"]
print(f"\n{'comparison':<32}{'DM-HLN':>9}{'p':>8}{'Clark-West':>13}{'p (one-sided)':>15}")
for label, small, large, nested in (("OLS-5 vs OLS-5 + text", "ols5", "ols5_plus_text", True),
                                    ("OLS-5 vs OLS-5 + TF-IDF ridge", "ols5", "ols5_plus_tfidf_ridge", True),
                                    ("random walk vs text alone", "random_walk", "text_alone", False),
                                    ("random walk vs OLS-5", "random_walk", "ols5", False)):
    dm = fc.diebold_mariano(err(small), err(large))
    if nested:
        cw = fc.clark_west(actual, runs[small].predictions["test"], runs[large].predictions["test"])
        print(f"{label:<32}{dm['statistic']:>9.2f}{dm['p_value']:>8.3f}{cw['statistic']:>13.2f}"
              f"{cw['p_value_one_sided']:>15.3f}")
    else:
        print(f"{label:<32}{dm['statistic']:>9.2f}{dm['p_value']:>8.3f}{'—':>13}{'—':>15}")
print("\nDiebold-Mariano is two-sided and a NEGATIVE statistic favours the first model; the Harvey-Leybourne-")
print("Newbold correction matters at n = 79, where the uncorrected normal over-rejects. Clark-West is")
print("one-sided and a POSITIVE statistic favours the larger model: under the null that the extra regressors")
print("are zero in population, the larger model still has to estimate them and is EXPECTED to lose, which is")
print("what makes DM mis-sized for a nested pair. Both are reported for every nested pair; neither alone.")

In [ ]:
order = [("random_walk", MUTED, "-"), ("ols5", TEAL, "-"), ("text_alone", TEAL, "--"),
         ("ols5_plus_text", RED, "-")]
plt.figure(figsize=(7, 3.2))
for key, colour, style in order:
    plt.plot(test_scores[key]["cumulative_squared_error"], color=colour, ls=style, lw=1.1,
             label=f"{runs[key].name} ({test_scores[key]['rmse']:.4f})")
plt.xlabel("test month, 2020-01 onward"); plt.ylabel("cumulative squared error")
plt.legend(frameon=False, fontsize=7.5); plt.tight_layout(); plt.show()
print("A gap produced entirely by one month is a different finding from one that accumulates steadily, and")
print("only the path shows which. Counting the months in which each model stands ahead of the random walk:")
walk = np.array(test_scores["random_walk"]["cumulative_squared_error"])
for key, _, _ in order[1:]:
    ahead = int((np.array(test_scores[key]["cumulative_squared_error"]) - walk < 0).sum())
    print(f"   {runs[key].name:<44}{ahead:>3} of {len(walk)} months")
print("OLS-5 plus text is behind from the first month and never draws level. Nothing here rests on March 2020.")

In [ ]:
llm = fc.llm_column(frame)
print("the LLM column, beside the structured ones and never inside them (protocol section 8.1)")
print(f"   {llm['n_scored']} statements scored, variants recorded: {llm['variants_present']}, "
      f"variant A only enters a forecast")
print(f"   coverage of the estimation sample: {100 * llm['coverage_of_estimation_sample']:.0f}%")
print(f"   pooled into model selection: {llm['pooled_into_model_selection']}")
print(f"   test RMSE {llm['test_metrics']['rmse']:.4f}, MAE {llm['test_metrics']['mae']:.4f}   "
      f"against the random walk's {test_scores['random_walk']['rmse']:.4f}")
print(f"   coefficients: " + ", ".join(f"{k} {v:+.4f}" for k, v in llm["coefficients"].items()))
print("\nNo model choice and no penalty in this block depended on that column. It is reported because it was")
print("recorded, and it is outside the comparison because the protocol put it there before it was scored.")

## F5. Reported as found

**Text does not help.** On the 79 test months from 2020-01, the random walk and AR(1) all but tie for the best forecast in the table;
adding the three text regressors to OLS-5 makes it substantially worse; the TF-IDF ridge does the same; text
alone beats OLS-5 but not the random walk, which is a statement about OLS-5, not about text. Clark–West — the
statistic built to give the larger model the benefit of the doubt — still favours the smaller one.

Every gate in this block passed. That is the design working, not the design failing: the block's three gates are
on procedure, and the protocol said in advance that this outcome passes all of them. A block whose gates could
only pass if text helped would be a block that could not report this result at all.

Three honest caveats, in the direction that weakens the finding rather than the claim:

- The window is 79 months and opens in January 2020, so a single monetary regime — the pandemic, the
  inflation surge and the tightening that answered it — is doing a great deal of the work. The cumulative
  path rules out one *month* driving the result; it does not rule out one *era*.
- The text features are three numbers — a seed-axis stance, a novelty index, and months since the last
  statement — plus a TF-IDF ridge. A richer representation is not ruled out by anything here.
- The comparator is strong. On this window the random walk all but ties AR(1), the best model Session 7 fitted,
  and beats the rest, so
  "text does not beat the bar" is partly a statement about how high the bar happens to sit.

What the block does establish is narrower and firmer: under a design fixed and hashed in advance, with the
sample restriction reported in its own column and the test window scored once, this text signal does not
improve this forecast.

# Your turn

**1. Chapter 13, exercise 5 — the eight FOMC sentences, on this lab's path.** The chapter's version asks for
FinBERT through `transformers`, which this offline machine does not carry, so run the comparison on the two
instruments the notebook does have. The eight sentences are printed as the chapter prints them, in the cell
below. (a) Score each one with the chapter's dictionary: positive words {robust, strong, appropriate}, negative
{softened, elevated, risks, tightening}, the score being the sum of the positive terms' TF-IDF weights minus
the sum of the negative terms', with the weights taken from **this lab's** training-window model (`model`), so
the IDF comes from 149 FOMC statements rather than from the eight sentences. (b) Score the same eight with
`tx.signal_decision_sentence` and with the block B hawk–dove axis. (c) Correlate the three. Where do they
disagree most? (d) Sentence 4 — *"ongoing increases in the target range will be appropriate"* — carries the
positive word "appropriate" and is a forward-looking hawkish signal. Sentiment and stance are not the same
dimension. Which of your three scores gets it right, and what would you have to build to get it right on
purpose rather than by luck?

**2. Chapter 13, exercise 6 — the two-model consistency diagnostic, as printed.** (a) Run
`book_code/ch13/yin_vu_persico_kappa.py` (reach it with `me.book_module("yin_vu_persico_kappa", chapter="ch13")`)
and report the mean, standard deviation and ratio of the two score sets, their Cohen κ computed from the
definition, and the Landis–Koch band it falls in. (b) Modify the script to introduce a controllable drift
parameter $\rho \in [0,1]$, with $\rho = 0$ two models that agree exactly and $\rho = 1$ two that are
independent; plot $\kappa(\rho)$, and read off the $\rho$ implied by the lowest pairwise agreement Yin, Vu and
Persico report, $\kappa = 0.36$. (c) Put each score set into a simplified difference-in-differences,
$y_{it} = \alpha + \beta\,\mathrm{Post}_t \cdot \mathrm{Exposure}_i + u_{it}$, report the two $\hat\beta$ and
their ratio, and — if the signs differ — write the minimum disclosure list that would have to accompany the
paper.

**3. Chapter 14, programming items 1–3, as printed.** (1★) Run `human_llm_kappa.py` and reproduce the overall
and per-class κ. Then move the rare class's share from 0.12 down to 0.03 and plot both κ curves against it: at
what share does the overall figure stop reflecting the failure of that class, and what does that imply for
reporting standards? (2★★) Run `bias_correction.py` and confirm the naive and corrected estimators' bias and
interval widths. Then (a) set the size-related systematic term in the measurement error to zero and watch the
naive estimator's bias, and (b) drop $c_i$ from the calibration equation and check whether the corrected
estimator is still approximately unbiased — and say why. (3★★) Run `prompt_permutation_test.py` and reproduce
the three presentation designs. Then, at a fixed total call budget, does spending on more items or on more
orderings per item reduce bias, and does it reduce variance? Give a design recommendation for a four-option
labelling task with 2,000 calls, and state the condition under which your recommendation changes.

**4. A fifth signal: a dictionary uncertainty index.** The four signals in this notebook are the TF-IDF change
series, the embedding hawk–dove axis, the decision-sentence regex, and the recorded language-model stance.
Build a fifth in the Baker–Bloom–Davis manner: count the uncertainty words {uncertain, uncertainty, risks,
downside, volatility, unusually, strains, turbulence} per statement, divide by the statement's token count, and
push the series through `tx.validate_signal` on the same subsequent policy path. Expect it to fail the sign
test, and say precisely what it is measuring that stance is not — then ask whether it should ever have been
asked to separate hiking from cutting cycles in the first place.

Chapter 13's exercises 2, 3 and 4 need files or packages this machine does not carry — `gensim` with
`code/ch13/earnings_calls_sample.txt`, `sentence-transformers` with `analyst_reports_sample.txt`, and `jieba`
with two PBoC reports downloaded from the web — so they are worked on a connected machine rather than here.

In [ ]:
CHAPTER13_EX5 = [
    "The Committee decided to raise the target range for the federal funds rate.",
    "Inflation remains elevated, reflecting supply and demand imbalances.",
    "Job gains have been robust in recent months.",
    "The Committee anticipates that ongoing increases in the target range will be appropriate.",
    "The pace of future increases will take into account cumulative tightening.",
    "Recent indicators suggest that spending and production have softened.",
    "The Committee is highly attentive to inflation risks.",
    "The labor market remains strong.",
]
POSITIVE = ("robust", "strong", "appropriate")
NEGATIVE = ("softened", "elevated", "risks", "tightening")

probe = [tx.Statement(date=f"2022-01-{i + 1:02d}", text=s, file="chapter13-exercise-5")
         for i, s in enumerate(CHAPTER13_EX5)]
W = model.transform(probe)                     # this lab's IDF, from the 149 training statements
for i, sentence in enumerate(CHAPTER13_EX5):
    pos = sum(W[i, model.column[w]] for w in POSITIVE if w in model.column)
    neg = sum(W[i, model.column[w]] for w in NEGATIVE if w in model.column)
    print(f"{i + 1}. dictionary {pos - neg:+.4f}   {sentence}")
print("\nStart here. Then add the other two scores, and write down what sentence 4 does to each of them.")

---
**What the gates were.** A: the corpus hashes and the line at 2014-12-31; the merges learned on that window; a topic model fitted on the same window whose top words reproduce the recorded run;
an IDF computed from 149 documents; and 9.A1's worked examples: six subword cuts that spell their phrase back, the
three documents' TF-IDF and cosines at the printed precision and the count matrix's zero share as recorded, the three-document topic model under three seeds. B: word vectors from the same 149, and both validation verdicts equal to
the recorded run — the one that fails and the one that passes, held to the same bound; 9.A2's noise draws, one word2vec step and its nine points' arithmetic at its printed precision; the rule narrowed to one sentence, as recorded. C: the chapter's
printed arithmetic; five file hashes, a parameter count and a state dictionary; a causal mask that is exactly
zero above the diagonal; a held-out split by whole statement with every character accounted for; outputs that reverse with the tokens until
positions or the mask are added; Machine B's first layer rebuilt by hand, and 9.A2's layer norm and feed-forward share; its next-character
scores and greedy text equal to the recorded run. D: four
provenance fields on every one of 389 replies, one prompt hash shared by A and D, and the same 48 statements
under every variant. E: 48 digits printed in chapter 14, reproduced by the chapter's own code at the chapter's
own precision. F: two file hashes, three frozen baselines to four decimals, splits of 250 / 60 / 79, an unfilled
gap at 2025-11-30, nothing after 2014-12-31 inside a fitted object, and a test window scored once that refuses
to be scored again.

None of them is a threshold on a result. Every one of them would pass unchanged if block B's axis had
validated, if block D's repeat had been the steadier of the two, and if block F's text had won.

In [ ]:
print(f"{len(GATES)} gates, {sum(ok for _, ok in GATES)} passing:")
print("   " + "  ".join(name for name, _ in GATES))
print(f"notebook run time: {time.time() - T_START:.0f} s")